In [ ]:
import pandas as pd
from source import image_id_converter as img_idc
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import einops as eo
import pathlib as pl

import matplotlib.cm as cm
from matplotlib import collections  as mc
from matplotlib import animation
%matplotlib inline

from scipy.stats import norm
from scipy.stats import entropy

import pandas as pd
import pickle
from PIL import Image
from time import time as timer
#import umap

from IPython.display import HTML
from IPython.display import Audio
import IPython

import tqdm.auto as tqdm

import torch
from torchvision import datasets, transforms
from torch import nn
from torch import optim
import torch.nn.functional as F

from torchvision import transforms

import sys
import einops as eo

from collections import defaultdict
import random


import os
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np
from pathlib import Path
import tempfile

from collections import defaultdict
import random


### Function to Check Image Size

In [ ]:
def analyze_image_sizes(image_directory, extensions=('.jpg', '.jpeg', '.tif', '.tiff')):
   """
   Analyze image sizes in a directory to help determine appropriate target_size.
   
   Args:
       image_directory (str): Path to directory containing images
       extensions (tuple): Allowed file extensions
       
   Returns:
       dict: Dictionary containing size analysis results
   """
   # Get all image paths
   image_paths = load_image_paths(image_directory, extensions)
   
   if not image_paths:
       print("No images found in directory")
       return None
   
   sizes = []
   widths = []
   heights = []
   failed_images = []
   
   print(f"Analyzing {len(image_paths)} images...")
   
   # Analyze each image
   for i, image_path in enumerate(image_paths):
       try:
           with Image.open(image_path) as img:
               width, height = img.size
               sizes.append((width, height))
               widths.append(width)
               heights.append(height)
       except Exception as e:
           failed_images.append((image_path, str(e)))
           print(f"Failed to read {image_path}: {e}")
       
       # Progress indicator for large datasets
       if (i + 1) % 100 == 0:
           print(f"Processed {i + 1}/{len(image_paths)} images...")
   
   if not sizes:
       print("No valid images found")
       return None
   
   # Calculate statistics
   unique_sizes = list(set(sizes))
   all_same_size = len(unique_sizes) == 1
   
   min_width = min(widths)
   max_width = max(widths)
   avg_width = sum(widths) / len(widths)
   
   min_height = min(heights)
   max_height = max(heights)
   avg_height = sum(heights) / len(heights)
   
   # Calculate quartiles
   import numpy as np
   width_q25, width_median, width_q75 = np.percentile(widths, [25, 50, 75])
   height_q25, height_median, height_q75 = np.percentile(heights, [25, 50, 75])
   
   min_size = (min_width, min_height)
   max_size = (max_width, max_height)
   avg_size = (avg_width, avg_height)
   
   # Create results dictionary
   results = {
       'total_images': len(image_paths),
       'valid_images': len(sizes),
       'failed_images': len(failed_images),
       'all_same_size': all_same_size,
       'unique_sizes_count': len(unique_sizes),
       'min_size': min_size,
       'max_size': max_size,
       'avg_size': avg_size,
       'min_width': min_width,
       'max_width': max_width,
       'avg_width': avg_width,
       'min_height': min_height,
       'max_height': max_height,
       'avg_height': avg_height,
       'width_q25': width_q25,
       'width_median': width_median,
       'width_q75': width_q75,
       'height_q25': height_q25,
       'height_median': height_median,
       'height_q75': height_q75,
       'failed_images': failed_images
   }
   
   # Print summary
   print("\n" + "="*50)
   print("IMAGE SIZE ANALYSIS SUMMARY")
   print("="*50)
   print(f"Total images found: {results['total_images']}")
   print(f"Valid images: {results['valid_images']}")
   print(f"Failed to read: {results['failed_images']}")
   print(f"\nAll images same size: {'Yes' if all_same_size else 'No'}")
   print(f"Number of unique sizes: {results['unique_sizes_count']}")
   
   print(f"\nSize ranges:")
   print(f"  Minimum size: {min_size[0]} x {min_size[1]}")
   print(f"  Maximum size: {max_size[0]} x {max_size[1]}")
   print(f"  Average size: {avg_size[0]:.1f} x {avg_size[1]:.1f}")
   
   print(f"\nWidth range: {min_width} - {max_width} (avg: {avg_width:.1f})")
   print(f"Width quartiles: Q25={width_q25:.1f}, Median={width_median:.1f}, Q75={width_q75:.1f}")
   print(f"Height range: {min_height} - {max_height} (avg: {avg_height:.1f})")
   print(f"Height quartiles: Q25={height_q25:.1f}, Median={height_median:.1f}, Q75={height_q75:.1f}")
   
   if results['failed_images']:
       print(f"\nFailed images:")
       for path, error in results['failed_images'][:5]:  # Show first 5 failures
           print(f"  {path}: {error}")
       if len(results['failed_images']) > 5:
           print(f"  ... and {len(results['failed_images']) - 5} more")
   
   # Suggest target size
   if all_same_size:
       print(f"\nRecommendation: Use target_size={min_size} (all images are the same size)")
   else:
       # Suggest a reasonable target size based on minimum dimensions
       suggested_size = min(min_width, min_height)
       # Round to common sizes
       common_sizes = [28, 32, 64, 128, 224, 256, 512]
       suggested_size = min(common_sizes, key=lambda x: abs(x - suggested_size))
       print(f"\nRecommendation: Consider target_size=({suggested_size}, {suggested_size})")
       print(f"  (Based on minimum dimension and common image sizes)")
   
   return results

### Functions to Get Image File Paths and Split them into Training and Validation Set

In [ ]:
def load_image_paths(image_directory, extensions=('.jpg', '.jpeg', '.tif', '.tiff')):
    """
    Load all image file paths from a directory.
    
    Args:
        image_directory (str): Path to directory containing images
        extensions (tuple): Allowed file extensions
        
    Returns:
        list: Sorted list of image file paths
    """
    image_paths = []
    image_dir = Path(image_directory)
    
    if not image_dir.exists():
        raise FileNotFoundError(f"Directory {image_directory} does not exist")
    
    for ext in extensions:
        # Find files with current extension (case insensitive)
        image_paths.extend(image_dir.glob(f"*{ext}"))
        image_paths.extend(image_dir.glob(f"*{ext.upper()}"))
    
    # Sort paths to ensure consistent ordering
    image_paths = sorted([str(path) for path in image_paths])
    
    print(f"Found {len(image_paths)} images in {image_directory}")
    return image_paths

In [ ]:

def split_train_validation(image_paths, labels, train_ratio=0.8, random_seed=42):
    """
    Split image paths into training and validation sets with balanced class distribution.
    
    Args:
        image_paths (list): List of image file paths
        labels (list): List of class labels corresponding to image paths
        train_ratio (float): Proportion for training set (default 0.8)
        random_seed (int): Random seed for reproducibility
        
    Returns:
        tuple: (train_paths, train_labels, val_paths, val_labels)
    """
    random.seed(random_seed)
    
    # Group paths by class
    class_groups = defaultdict(list)
    for path, label in zip(image_paths, labels):
        class_groups[label].append(path)
    
    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    
    # Split each class maintaining proportion
    for class_label, paths in class_groups.items():
        # Shuffle paths for this class
        shuffled_paths = paths.copy()
        random.shuffle(shuffled_paths)
        
        # Calculate split point
        n_train = int(len(shuffled_paths) * train_ratio)
        
        # Split paths
        class_train_paths = shuffled_paths[:n_train]
        class_val_paths = shuffled_paths[n_train:]
        
        # Add to final lists
        train_paths.extend(class_train_paths)
        train_labels.extend([class_label] * len(class_train_paths))
        val_paths.extend(class_val_paths)
        val_labels.extend([class_label] * len(class_val_paths))
    
    # Final shuffle to mix classes
    combined_train = list(zip(train_paths, train_labels))
    combined_val = list(zip(val_paths, val_labels))
    random.shuffle(combined_train)
    random.shuffle(combined_val)
    
    train_paths, train_labels = zip(*combined_train) if combined_train else ([], [])
    val_paths, val_labels = zip(*combined_val) if combined_val else ([], [])
    
    print(f"Training set: {len(train_paths)} images")
    print(f"Validation set: {len(val_paths)} images")
    
    return list(train_paths), list(train_labels), list(val_paths), list(val_labels)

### Functions Data Loading Pipeline

In [ ]:
def convert_to_grayscale(image):
    """
    Convert PIL image to grayscale.
    
    Args:
        image (PIL.Image): Input image
        
    Returns:
        PIL.Image: Grayscale image
    """
    return image.convert('L')


def apply_aging_effect(image):
    """
    Apply aging effects to PIL image (adapted from your existing function).
    
    Args:
        image (PIL.Image): Input image
        
    Returns:
        PIL.Image: Aged image
    """
    # Ensure image is in RGB mode for aging effects
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Heavy JPEG compression using temporary file
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_file:
        temp_path = temp_file.name
        image.save(temp_path, 'JPEG', quality=15)
        image = Image.open(temp_path)
        os.unlink(temp_path)  # Clean up temp file
    
    # Significant brightness reduction
    enhancer = ImageEnhance.Brightness(image)
    image = enhancer.enhance(0.8)
    
    # Low contrast
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(0.9)
    
    # Add significant noise
    img_array = np.array(image)
    noise = np.random.normal(0, 0.1, img_array.shape).astype(np.uint8)
    img_array = np.clip(img_array.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    image = Image.fromarray(img_array)
    
    # Strong blur
    image = image.filter(ImageFilter.GaussianBlur(radius=1.5))
    
    return image


def load_and_preprocess_image(image_path, target_size=(28, 28), convert_grayscale=True, apply_aging=False):
    """
    Load and preprocess a single image.
    
    Args:
        image_path (str): Path to image file
        target_size (tuple): Target size for resizing (width, height)
        convert_grayscale (bool): Whether to convert to grayscale
        apply_aging (bool): Whether to apply aging effects
        
    Returns:
        PIL.Image: Preprocessed image
    """
    try:
        # Load image
        image = Image.open(image_path)
        
        # Apply aging effects first (works best on RGB images)
        if apply_aging:
            image = apply_aging_effect(image)
        
        # Convert to grayscale if requested
        if convert_grayscale:
            image = convert_to_grayscale(image)
        
        # Resize to target size
        image = image.resize(target_size, Image.Resampling.LANCZOS)
        
        return image
        
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None


def create_transforms(mean=0.5, std=1.0):
    """
    Create image transforms matching your MNIST setup.
    
    Args:
        mean (float): Normalization mean
        std (float): Normalization standard deviation
        
    Returns:
        torchvision.transforms.Compose: Transform pipeline
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((mean,), (std,))
    ])
    return transform


def create_label_transform():
    """
    Create label transform matching your MNIST setup.
    
    Returns:
        torchvision.transforms.Compose: Label transform pipeline
    """
    label_transform = transforms.Compose([
        lambda x: torch.LongTensor([x])
    ])
    return label_transform




class CustomImageDataset(Dataset):
    """
    Custom dataset class that exactly mimics torchvision.datasets.MNIST structure.
    
    This dataset replicates ALL MNIST properties including .data and .targets tensors.
    """
    
    def __init__(self, image_paths, labels, target_size=(28, 28), 
                 convert_grayscale=True, apply_aging=False,
                 transform=None, target_transform=None):
        """
        Initialize the dataset with MNIST-compatible structure.
        
        Args:
            image_paths (list): List of image file paths
            labels (list): List of integer labels corresponding to images
            target_size (tuple): Target size for resizing images
            convert_grayscale (bool): Whether to convert images to grayscale
            apply_aging (bool): Whether to apply aging effects
            transform (callable): Transform to apply to images
            target_transform (callable): Transform to apply to labels
        """
        self.image_paths = image_paths
        self.target_size = target_size
        self.convert_grayscale = convert_grayscale
        self.apply_aging = apply_aging
        self.transform = transform
        self.target_transform = target_transform
        
        # Validate that we have equal number of images and labels
        if len(image_paths) != len(labels):
            raise ValueError(f"Number of images ({len(image_paths)}) must match number of labels ({len(labels)})")
        
        # Create MNIST-compatible attributes
        self.targets = torch.tensor(labels, dtype=torch.long)  # Shape: [N]
        self._data = None  # Will be loaded when first accessed
        
        print(f"Initializing dataset with {len(image_paths)} images...")
    
    @property
    def data(self):
        """
        MNIST-compatible data property that returns all images as a tensor.
        Shape: [N, H, W] for grayscale or [N, H, W, C] for color
        Dtype: torch.uint8 (same as MNIST)
        
        Images are loaded and cached on first access.
        """
        if self._data is None:
            print("Loading all images into .data tensor (this may take a moment)...")
            self._load_all_images()
        return self._data
    
    def _load_all_images(self):
        """Load all images into the .data tensor."""
        all_images = []
        
        for i, image_path in enumerate(self.image_paths):
            # Load and preprocess image
            image = load_and_preprocess_image(
                image_path,
                target_size=self.target_size,
                convert_grayscale=self.convert_grayscale,
                apply_aging=self.apply_aging
            )
            
            # Handle failed loading
            if image is None:
                if self.convert_grayscale:
                    image = Image.new('L', self.target_size, 0)
                else:
                    image = Image.new('RGB', self.target_size, (0, 0, 0))
            
            # Convert to numpy array with uint8 dtype (like MNIST)
            img_array = np.array(image, dtype=np.uint8)
            all_images.append(img_array)
            
            # Progress indicator
            if (i + 1) % 100 == 0:
                print(f"Loaded {i + 1}/{len(self.image_paths)} images...")
        
        # Stack all images and convert to tensor
        all_images = np.stack(all_images, axis=0)
        self._data = torch.from_numpy(all_images)
        
        print(f"Loaded .data tensor with shape: {self._data.shape}, dtype: {self._data.dtype}")
    
    def __len__(self):
        """Return the total number of samples."""
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        """
        Get a single sample from the dataset.
        
        Args:
            idx (int): Index of the sample to retrieve
            
        Returns:
            tuple: (image, label) where image is a tensor and label is a tensor
            
        Note: This applies transforms to the raw data, just like MNIST
        """
        # Get raw image from .data tensor (this will load all images if needed)
        raw_image = self.data[idx]  # Shape: [H, W] or [H, W, C]
        
        # Convert tensor back to PIL Image for transforms
        if raw_image.dim() == 2:  # Grayscale
            image = Image.fromarray(raw_image.numpy(), mode='L')
        else:  # Color
            image = Image.fromarray(raw_image.numpy(), mode='RGB')
        
        # Get label from targets tensor
        label = self.targets[idx].item()  # Convert to Python int
        
        # Apply transforms (same as MNIST)
        if self.transform:
            image = self.transform(image)
        
        if self.target_transform:
            label = self.target_transform(label)
        
        return image, label




def create_image_dataset(image_paths, labels, target_size=(28, 28),
                        convert_grayscale=True, apply_aging=False,
                        mean=0.5, std=1.0):
    """
    Create a complete image dataset ready for use with DataLoader.
    
    Args:
        image_paths (list): List of image file paths
        labels (list): List of integer labels for the images
        target_size (tuple): Target size for resizing images
        convert_grayscale (bool): Whether to convert images to grayscale
        apply_aging (bool): Whether to apply aging effects
        mean (float): Normalization mean
        std (float): Normalization standard deviation
        
    Returns:
        CustomImageDataset: Dataset ready for use with DataLoader
    """
    # Step 1: Create transforms
    transform = create_transforms(mean=mean, std=std)
    target_transform = create_label_transform()
    
    # Step 2: Create dataset
    dataset = CustomImageDataset(
        image_paths=image_paths,
        labels=labels,
        target_size=target_size,
        convert_grayscale=convert_grayscale,
        apply_aging=apply_aging,
        transform=transform,
        target_transform=target_transform
    )
    
    return dataset



In [ ]:
def collate_ae_dataset(samples):
    """
    The function collates sampels into a batch, and creates noisy samples if DENOISING is True
    for the denoising autoencoder.
    """
    xs = [s[0] for s in samples]
    ys = [s[1] for s in samples]
    #Extracts the first element (input data) from each sample into list xs.
    #Extracts the second element (labels or targets) from each sample into list ys.
    #This assumes each sample is a tuple or list with at least two elements.
    
    xs = torch.stack(xs)
    ys = torch.concat(ys)
    #torch.stack(xs) combines the list of input tensors into a single 
    #tensor along a new dimension (creating a batch dimension).
    #torch.concat(ys) concatenates the label tensors along 
    #the existing first dimension. This suggests the labels might have 
    #variable lengths or already include a batch-like dimension.

    add_noise = NOISE_RATE > 0.
    #Checks if noise should be added based on a global variable NOISE_RATE.
    #If NOISE_RATE is greater than 0, noise will be added to the inputs.
    
    if add_noise:
      sh = xs.shape
      noise_mask = torch.bernoulli(torch.full(sh, NOISE_RATE))  # 0 (keep) or 1 (replace with noise)
      #Gets the shape of the input tensor batch.
      #Creates a binary mask using Bernoulli sampling, where each element has NOISE_RATE probability of being 1 
      #(indicating where noise will be applied) and 1-NOISE_RATE probability of being 0.
            
      sp_noise = torch.bernoulli(torch.full(sh, 0.5))-0.5  # -1 or 1
      #Generates the actual noise values as either -0.5 or 0.5.
      #First creates a tensor of the same shape filled with 0.5, then applies Bernoulli 
      #sampling to get 0s or 1s.
      #Subtracts 0.5 to convert to -0.5 or 0.5 (this creates salt and pepper noise).
        
      xns = xs * (1-noise_mask) + sp_noise * noise_mask
      #Creates the noisy input xns by:
          #Keeping original values where the mask is 0: xs * (1-noise_mask)
          #Adding noise values where the mask is 1: sp_noise * noise_mask
          #The result is a tensor where some values are preserved 
          #from the original input and others are replaced with noise.
      
      # sp = sp_noise
    else:
       xns = xs
    #If no noise is to be added, the noisy input is the same as the original input.

    return xns.to(device), xs.to(device), ys.to(device)
    #Returns three tensors, all moved to the specified device (likely GPU):

    #xns: The inputs with noise added (or original inputs if no noise)
    #xs: The original clean inputs
    #ys: The labels or targets
    #
    #
    #This return structure is typical for denoising autoencoders, where you need 
    #both the noisy input (fed to the encoder) and the clean target 
    #(used to compute the reconstruction loss).
    #
    #This function is specifically designed for training denoising autoencoders, 
    #where the model learns to remove noise from corrupted inputs by trying 
    #to reconstruct the original clean data.

### Functions to Prepare and Execute Training

In [ ]:
def get_samples(valid_loader):
  # 1. get numpy array of all validation images:
  val_images_noisy = []
  val_images = []
  val_labels = []

  for batch_idx, (noisy_data, data, target) in enumerate(valid_loader):
      val_images_noisy.append(noisy_data.detach().cpu().numpy())
      val_images.append(data.detach().cpu().numpy())
      val_labels.append(target.detach().cpu().numpy())

  val_images_noisy = np.concatenate(val_images_noisy, axis=0)
  val_images = np.concatenate(val_images, axis=0)
  val_labels = np.concatenate(val_labels, axis=0)

  # 2. get numpy array of balanced validation samples for visualization:
  sample_images_noisy = []
  sample_images = []
  sample_labels = []
  single_el_idx = []  # indexes of single element per class for visualization

  n_class = np.max(val_labels) + 1
  # Determines the number of classes (for MNIST, this would be 10, representing digits 0-9).
  for class_idx in range(n_class):
    map_c = val_labels == class_idx

    ims_c_noisy = val_images_noisy[map_c]
    ims_c = val_images[map_c]
    print('class label:')
    print(class_idx)
    print('shape selected class')
    print(ims_c.shape)
    # For each class:
       # Creates a boolean mask map_c identifying all samples of the current class.
       # Extracts noisy and clean images for just this class.
      

    samples_idx = np.random.choice(len(ims_c), N_SAMPLE, replace=False)

    ims_c_noisy_samples = ims_c_noisy[samples_idx]
    ims_c_samples = ims_c[samples_idx]
    # Randomly selects N_SAMPLE images from the current class.
    # replace=False ensures no duplicates are selected.
    # Extracts both noisy and clean versions of these sampled images.
      

    sample_images_noisy.append(ims_c_noisy_samples)
    sample_images.append(ims_c_samples)

    sample_labels.append([class_idx]*N_SAMPLE)

    # Adds the sampled noisy images, clean images, and labels to their respective lists.
    # Creates an array of N_SAMPLE repeated labels for this class.

    start_idx = N_SAMPLE*class_idx
    single_el_idx.extend([start_idx + i for i in range(min(N_VIS_SAMPLE, N_SAMPLE))])
    # Calculates the indices for the first N_VIS_SAMPLE elements of this class in the final concatenated array.
    # These indices will be used to extract a smaller subset for visualization.

    
  sample_images_noisy = np.concatenate(sample_images_noisy, axis=0)
  sample_images = np.concatenate(sample_images, axis=0)
  sample_labels = np.concatenate(sample_labels, axis=0)
  single_el_idx = np.array(single_el_idx)
  #Combines all class samples into single arrays.
  #Converts the index list to a NumPy array.

  samples = {
      'images_noisy': sample_images_noisy,
      'images': sample_images,
      'labels': sample_labels,
      'single_el_idx': single_el_idx

  }
  return samples
# Creates and returns a dictionary with all collected samples.


# This function ensures we have:
# 
# A balanced number of samples for each class (equal representation)
# Both noisy and clean versions of each image
# A mapping between the noisy and clean versions
# A subset of indices for visualization purposes
# This is particularly useful for creating visualizations that show how the model behaves across different classes, or for comparing reconstruction quality across digits.



In [ ]:
def get_samples_from_dataset(valid_dataset):
    """
    Modified version that extracts samples directly from valid_dataset instead of valid_loader.
    Includes image_paths in the returned samples object.
    """
    # 1. Get numpy arrays of all validation data from dataset
    val_images = valid_dataset.data.cpu().numpy()  # Shape: [N, H, W]
    val_labels = valid_dataset.targets.cpu().numpy()  # Shape: [N]
    val_image_paths = valid_dataset.image_paths  # List of strings
    
    # Add channel dimension if needed (assumes grayscale)
    if val_images.ndim == 3:
        val_images = val_images[:, np.newaxis, :, :]  # [N, 1, H, W]
    
    # Apply transforms to get processed versions (matching what loader would do)
    val_images_transformed = []
    for i in range(len(valid_dataset)):
        img_tensor, _ = valid_dataset[i]  # Get transformed image
        val_images_transformed.append(img_tensor.cpu().numpy())
    val_images_transformed = np.array(val_images_transformed)
    
    # Create noisy versions (matching collate_ae_dataset behavior)
    val_images_noisy = add_salt_pepper_noise(val_images_transformed)
    
    # 2. Get numpy arrays of balanced validation samples for visualization
    sample_images_noisy = []
    sample_images = []
    sample_labels = []
    sample_image_paths = []
    single_el_idx = []
    
    n_class = np.max(val_labels) + 1
    
    for class_idx in range(n_class):
        map_c = val_labels == class_idx
        ims_c_noisy = val_images_noisy[map_c]
        ims_c = val_images_transformed[map_c]
        paths_c = [val_image_paths[i] for i in range(len(val_image_paths)) if map_c[i]]
        
        print('class label:')
        print(class_idx)
        print('shape selected class')
        print(ims_c.shape)
        
        # Randomly select N_SAMPLE images from the current class
        samples_idx = np.random.choice(len(ims_c), N_SAMPLE, replace=False)
        ims_c_noisy_samples = ims_c_noisy[samples_idx]
        ims_c_samples = ims_c[samples_idx]
        paths_c_samples = [paths_c[i] for i in samples_idx]
        
        sample_images_noisy.append(ims_c_noisy_samples)
        sample_images.append(ims_c_samples)
        sample_labels.append([class_idx] * N_SAMPLE)
        sample_image_paths.extend(paths_c_samples)
        
        start_idx = N_SAMPLE * class_idx
        single_el_idx.extend([start_idx + i for i in range(min(N_VIS_SAMPLE, N_SAMPLE))])
    
    sample_images_noisy = np.concatenate(sample_images_noisy, axis=0)
    sample_images = np.concatenate(sample_images, axis=0)
    sample_labels = np.concatenate(sample_labels, axis=0)
    sample_image_paths = np.array(sample_image_paths)  # Convert to numpy array
    single_el_idx = np.array(single_el_idx)
    
    samples = {
        'images_noisy': sample_images_noisy,
        'images': sample_images,
        'labels': sample_labels,
        'image_paths': sample_image_paths,  # NEW: Added image paths
        'single_el_idx': single_el_idx
    }
    
    return samples


def get_samples_from_dataset_all(valid_dataset):
    """
    Modified version that extracts samples directly from valid_dataset instead of valid_loader.
    Includes image_paths in the returned samples object.
    """
    # 1. Get numpy arrays of all validation data from dataset
    val_images = valid_dataset.data.cpu().numpy()  # Shape: [N, H, W]
    val_labels = valid_dataset.targets.cpu().numpy()  # Shape: [N]
    val_image_paths = valid_dataset.image_paths  # List of strings
    
    # Add channel dimension if needed (assumes grayscale)
    if val_images.ndim == 3:
        val_images = val_images[:, np.newaxis, :, :]  # [N, 1, H, W]
    
    # Apply transforms to get processed versions (matching what loader would do)
    val_images_transformed = []
    for i in range(len(valid_dataset)):
        img_tensor, _ = valid_dataset[i]  # Get transformed image
        val_images_transformed.append(img_tensor.cpu().numpy())
    val_images_transformed = np.array(val_images_transformed)
    
    # Create noisy versions (matching collate_ae_dataset behavior)
    val_images_noisy = add_salt_pepper_noise(val_images_transformed)
    
    # 2. Get numpy arrays of balanced validation samples for visualization
   # sample_images_noisy = []
   # sample_images = []
   # sample_labels = []
   # sample_image_paths = []
   # single_el_idx = []
   # 
   # n_class = np.max(val_labels) + 1
   # 
   # for class_idx in range(n_class):
   #     map_c = val_labels == class_idx
   #     ims_c_noisy = val_images_noisy[map_c]
   #     ims_c = val_images_transformed[map_c]
   #     paths_c = [val_image_paths[i] for i in range(len(val_image_paths)) if map_c[i]]
   #     
   #     print('class label:')
   #     print(class_idx)
   #     print('shape selected class')
   #     print(ims_c.shape)
   #     
   #     # Randomly select N_SAMPLE images from the current class
   #     samples_idx = np.random.choice(len(ims_c), N_SAMPLE, replace=False)
   #     ims_c_noisy_samples = ims_c_noisy[samples_idx]
   #     ims_c_samples = ims_c[samples_idx]
   #     paths_c_samples = [paths_c[i] for i in samples_idx]
   #     
   #     sample_images_noisy.append(ims_c_noisy_samples)
   #     sample_images.append(ims_c_samples)
   #     sample_labels.append([class_idx] * N_SAMPLE)
   #     sample_image_paths.extend(paths_c_samples)
   #     
   #     start_idx = N_SAMPLE * class_idx
   #     single_el_idx.extend([start_idx + i for i in range(min(N_VIS_SAMPLE, N_SAMPLE))])
   # 
    #sample_images_noisy = np.concatenate(sample_images_noisy, axis=0)
    sample_images_noisy = val_images_noisy
    #sample_images = np.concatenate(sample_images, axis=0)
    sample_images = val_images_transformed
    #sample_labels = np.concatenate(sample_labels, axis=0)
    sample_labels = val_labels
    sample_image_paths = np.array(val_image_paths)  # Convert to numpy array
    single_el_idx = np.array(list(range(0, len(sample_images))))
    
    samples = {
        'images_noisy': sample_images_noisy,
        'images': sample_images,
        'labels': sample_labels,
        'image_paths': sample_image_paths,  # NEW: Added image paths
        'single_el_idx': single_el_idx
    }
    
    return samples




def add_salt_pepper_noise(images, noise_amount=0.05):
    """
    Helper function to add salt and pepper noise (matching collate_ae_dataset behavior).
    """
    noisy = images.copy()
    # Add salt (white pixels)
    salt_mask = np.random.random(images.shape) < noise_amount / 2
    noisy[salt_mask] = 1.0
    # Add pepper (black pixels)
    pepper_mask = np.random.random(images.shape) < noise_amount / 2
    noisy[pepper_mask] = -0.5
    return noisy

In [ ]:
def to_np_showable(pt_img):
  np_im = pt_img.detach().cpu().numpy()
  if len(np_im.shape) == 4:
    np_im = np_im[0]

  if np_im.shape[0] > 3:
    np_im = np_im[-3:]

  return (eo.rearrange(np_im, 'c h w -> h w c')/2+.5).clip(0., 1.)

#This function converts a PyTorch tensor image to a NumPy array suitable for visualization.
#pt_img.detach().cpu().numpy() - Detaches the tensor from the computation graph, moves it to CPU if it's on GPU, and converts it to a NumPy array.
#if len(np_im.shape) == 4: - Checks if the image has a batch dimension (shape: [batch, channels, height, width]).
#np_im = np_im[0] - If there's a batch dimension, takes only the first image in the batch.
#if np_im.shape[0] > 3: - Checks if there are more than 3 channels.
#np_im = np_im[-3:] - If there are more than 3 channels, keeps only the last 3 channels (useful for handling multi-channel data).
#eo.rearrange(np_im, 'c h w -> h w c') - Uses the einops library to rearrange the tensor from PyTorch's [channels, height, width] format to matplotlib's [height, width, channels] format.
#/2+.5 - Applies normalization assuming the image data is in the range [-1, 1], converting it to [0, 1].
#.clip(0., 1.) - Ensures all values are within the [0, 1] range, clamping any values outside this range.

def plot_im(im, is_torch=True):
  plt.imshow(to_np_showable(im) if is_torch else im, cmap='gray')
  plt.show()
  plt.close()

#This function plots a single image.
#is_torch=True - Default parameter indicating whether the input is a PyTorch tensor.
#to_np_showable(im) if is_torch else im - Converts the image to a NumPy array if it's a PyTorch tensor, otherwise uses it directly.
#plt.imshow(..., cmap='gray') - Displays the image using matplotlib with a grayscale colormap.
#plt.show() - Renders the plot.
#plt.close() - Closes the figure to free up memory.

def plot_im_samples(ds, n=5, is_torch=False):
  fig, axs = plt.subplots(1, n, figsize=(16, n))
  for i, image in enumerate(ds[:n]):
      axs[i].imshow(to_np_showable(image) if is_torch else image, cmap='gray')
      axs[i].set_axis_off()
  plt.show()
  plt.close()

#This function plots multiple images from a dataset in a row.
#ds - The dataset or collection of images to sample from.
#n=5 - Default number of images to display.
#is_torch=False - Default parameter indicating whether the inputs are PyTorch tensors.
#plt.subplots(1, n, figsize=(16, n)) - Creates a figure with a single row of n subplots, with a width of 16 inches and height of n inches.
#The loop iterates through the first n images in the dataset:
#
#axs[i].imshow(...) - Displays each image in its corresponding subplot.
#axs[i].set_axis_off() - Removes axis labels and ticks for cleaner visualization.
#
#
#plt.show() - Renders the entire plot with all images.
#plt.close() - Closes the figure to free up memory.

In [ ]:
# get mean and std of an array with numpy:
def get_mean_std(x):
    x_mean = np.mean(x)
    x_std = np.std(x)
    return x_mean, x_std

# get min and max of an array with numpy:
def get_min_max(x):
    x_min = np.min(x)
    x_max = np.max(x)
    return x_min, x_max

def is_iterable(obj):
    try:
        iter(obj)
    except Exception:
        return False
    else:
        return True

#This function checks if an object is iterable (can be looped over).
#It uses a try-except block to attempt to call iter(obj), which will succeed only if obj is iterable.
#If calling iter(obj) raises any exception, the function returns False.
#If no exception occurs, the function returns True.

def type_len(obj):
    t = type(obj)
    if is_iterable(obj):
        sfx = f', shape: {obj.shape}' if t == np.ndarray else ''
        print(f'type: {t}, len: {len(obj)}{sfx}')
    else:
        print(f'type: {t}, len: {len(obj)}')

#This is a utility function for debugging that prints information about an object.
#t = type(obj) - Gets the type of the provided object.
#It checks if the object is iterable using the is_iterable function defined earlier.
#If the object is iterable:
#
#It checks if the object is a NumPy array (t == np.ndarray).
#If it's a NumPy array, it adds shape information to the output string.
#It prints the type and length of the object, along with shape information if applicable.
#
#
#If the object is not iterable, it still attempts to print the type and length (though this might raise an error if len() isn't applicable to the object).
#
#Note: There seems to be an issue with the type_len function - it tries to call len() on non-iterable objects in the else clause, 
#which would typically cause an error. This might be a bug in the code.


In [ ]:
# merging 2d matrix of images in 1 image
def mosaic(mtr_of_ims):
  ny = len(mtr_of_ims)
  assert(ny != 0)
  #Gets the number of rows in the matrix and asserts that it's not empty.

  nx = len(mtr_of_ims[0])
  assert(nx != 0)
  #Gets the number of columns in the first row and asserts that it's not empty.

  im_sh = mtr_of_ims[0][0].shape

  assert (2 <= len(im_sh) <= 3)
  #Gets the shape of the first image in the matrix.
  #Verifies that the image is either 2D (grayscale) or 3D (with channels).
    
  multichannel = len(im_sh) == 3

  if multichannel:
    h, w, c = im_sh
  else:
    h, w = im_sh
  #Determines if the images have multiple channels.
  #If multichannel, unpacks height, width, and channels. Otherwise, just height and width.

  h_c = h * ny + 1 * (ny-1)
  w_c = w * nx + 1 * (nx-1)
  #Calculates the total height and width of the canvas.
  #Adds 1 pixel spacing between images (both horizontally and vertically).

  canv_sh = (h_c, w_c, c) if multichannel else (h_c, w_c)
  canvas = np.ones(shape=canv_sh, dtype=np.float32)*0.5
  #Defines the shape of the canvas based on whether images are multichannel.
  #Creates a canvas filled with gray (0.5) values, assuming image values are in [0,1] range.

  for iy, row in enumerate(mtr_of_ims):
    y_ofs = iy * (h + 1)
    #Loops through each row of images.
    #Calculates the vertical offset for the current row.
    for ix, im in enumerate(row):
      x_ofs = ix * (w + 1)
      #Loops through each image in the current row.
      #Calculates the horizontal offset for the current image.
      canvas[y_ofs:y_ofs + h, x_ofs:x_ofs + w] = im
      #Copies the current image to the appropriate position in the canvas.
      #This uses NumPy's array slicing to place the image at the correct location.
  return canvas

In [ ]:
from collections import defaultdict
import random

def split_train_validation(image_paths, labels, train_ratio=0.8, random_seed=42):
    """
    Split image paths into training and validation sets with balanced class distribution.
    
    Args:
        image_paths (list): List of image file paths
        labels (list): List of class labels corresponding to image paths
        train_ratio (float): Proportion for training set (default 0.8)
        random_seed (int): Random seed for reproducibility
        
    Returns:
        tuple: (train_paths, train_labels, val_paths, val_labels)
    """
    random.seed(random_seed)
    
    # Group paths by class
    class_groups = defaultdict(list)
    for path, label in zip(image_paths, labels):
        class_groups[label].append(path)
    
    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    
    # Split each class maintaining proportion
    for class_label, paths in class_groups.items():
        # Shuffle paths for this class
        shuffled_paths = paths.copy()
        random.shuffle(shuffled_paths)
        
        # Calculate split point
        n_train = int(len(shuffled_paths) * train_ratio)
        
        # Split paths
        class_train_paths = shuffled_paths[:n_train]
        class_val_paths = shuffled_paths[n_train:]
        
        # Add to final lists
        train_paths.extend(class_train_paths)
        train_labels.extend([class_label] * len(class_train_paths))
        val_paths.extend(class_val_paths)
        val_labels.extend([class_label] * len(class_val_paths))
    
    # Final shuffle to mix classes
    combined_train = list(zip(train_paths, train_labels))
    combined_val = list(zip(val_paths, val_labels))
    random.shuffle(combined_train)
    random.shuffle(combined_val)
    
    train_paths, train_labels = zip(*combined_train) if combined_train else ([], [])
    val_paths, val_labels = zip(*combined_val) if combined_val else ([], [])
    
    print(f"Training set: {len(train_paths)} images")
    print(f"Validation set: {len(val_paths)} images")
    
    return list(train_paths), list(train_labels), list(val_paths), list(val_labels)

### Functions To Analyse Results

In [ ]:
def plot_hist(history, logscale=True):
    """
    plot training loss
    """

    loss = history['loss']
    v_loss = history['val_loss']
    epochs = history['epoch']

    # This function visualizes training history (loss over epochs).
    # Extracts training loss, validation loss, and epoch numbers from the history dictionary.

    
    plot = plt.semilogy if logscale else plt.plot
    # Cleverly chooses between logarithmic scale (plt.semilogy) or linear scale (plt.plot) based on the logscale parameter.
    # Default is logarithmic scale, which is often better for visualizing loss curves as they typically decrease exponentially.
    
    plot(epochs, loss, label='training');
    plot(epochs, v_loss, label='validation');
    # Plots both training and validation loss curves using the selected plotting function.
    # Labels each curve for the legend.
    
    plt.legend()
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.show()
    plt.close()
    # Adds a legend, axis labels, displays the plot, and then closes the figure.



def plot_samples(sample_history, samples, epoch_stride=5, fig_scale=1):
    """
    Plots input, noisy samples (for DAE) and reconstruction.
    Each `epoch_stride`-th epoch
    """
    # This function visualizes sample reconstructions over training epochs.
    # Shows how the model's reconstruction capability improves over time.

    single_el_idx = samples['single_el_idx']
    images_noisy = samples['images_noisy'][single_el_idx, 0]
    images = samples['images'][single_el_idx, 0]
    # Extracts indices for selected samples to visualize.
    # Gets the noisy input images and the original clean images for these samples.
    # The , 0 indexing suggests selecting the first channel of each image.

    last_epoch = np.max(list(sample_history.keys()))
    # Determines the last epoch number in the history data.

    for epoch_idx, hist_el in sample_history.items():
      if epoch_idx % epoch_stride != 0 and epoch_idx != last_epoch:
        continue
    # Iterates through each epoch's results in the history.
    # Uses epoch_stride to select only every nth epoch (to avoid too many visualizations).
    # Always includes the last epoch regardless of the stride.

      samples_arr = [images_noisy, hist_el['y'][single_el_idx, 0], images]
    # Creates an array of three sets of images to visualize side by side:
       # The noisy input images
       # The model's reconstructions for the current epoch
       # The original clean images (ground truth)

      ny = len(samples_arr)
      nx = len(samples_arr[0])

      plt.figure(figsize=(fig_scale*nx, fig_scale*ny))
      # Calculates the dimensions of the visualization grid.
      # Creates a figure with size proportional to the number of samples.

        
      m = mosaic(samples_arr)
      # Uses the previously defined mosaic function to create a grid of all images.

      plt.title(f'after epoch {int(epoch_idx)}')
      plt.imshow(m, cmap='gray', vmin=-.5, vmax=.5)
      # Adds a title showing which epoch this visualization represents.
      # Displays the mosaic with a grayscale colormap and fixed value range.
      # The vmin=-.5, vmax=.5 matches the normalized data range we've seen before.

        
      plt.tight_layout(pad=0.1, h_pad=0, w_pad=0)
      plt.show()
      plt.close()
      # Ensures proper spacing in the figure.
      # Displays the figure and then closes it to free memory.

# This function creates a powerful visualization showing the progression of the model's reconstruction ability across epochs. Each visualization has three rows:
# 
# The noisy inputs
# The model's reconstructions
# The original clean images (targets)
# 
# This makes it easy to see how the model gradually learns to denoise and reconstruct the images over the course of training.

In [ ]:
# These are utility functions for working with trained models at different stages of training. Let me break them down:

def run_on_trained(model, root_dir, run_fn, ep=None, model_filename=None):
    """
    Helper function to excecute any function on model in state after `ep` training epoch
    """
    # This function loads a model checkpoint and runs a specified function on it.
    # Parameters:
    # 
    # model: The neural network model instance
    # root_dir: Directory containing saved model checkpoints
    # run_fn: The function to run on the loaded model
    # ep: Specific epoch to load (optional)
    # model_filename: Specific checkpoint file to load (optional)

    if model_filename is None:
        if ep is not None:
            model_filename = root_dir/f'model_{ep:03d}.pth'
        else:
            model_filename = sorted(list(root_dir.glob('*.pth')))[-1]  # last model state
    # Determines which model checkpoint file to load:
    # 
    # If a specific filename is provided, use that (in this case this code block would be skipped)
    # If an epoch number is provided, construct the filename using a pattern
    # If neither is provided, use the last checkpoint file (by alphabetical sorting)
    # The code uses pathlib's Path objects for file handling (using / for path joining)

    
    model_dict = torch.load(model_filename,weights_only=False)

    model.load_state_dict(model_dict['model_state_dict'])

    # Loads the saved model state from the specified file
    # The weights_only=False parameter indicates to load the full state dictionary (not just weights)
    # Restores the model parameters from the saved state dictionary
    

    run_fn(model)
    # Calls the provided function on the loaded model

def run_on_all_training_history(model, root_dir, run_fn, n_ep=None):
    """
    Helper function to excecute any function on model state after each of the training epochs
    """
    # This function runs a specified function on multiple model checkpoints from different training epochs.
    # Parameters:
    # 
    # model: The neural network model instance
    # root_dir: Directory containing saved model checkpoints
    # run_fn: The function to run on each loaded model state
    # n_ep: Specific number of epochs to process (optional)
    
    if n_ep is not None:
        for ep in range(n_ep):
            print(f'running on epoch {ep+1}/{n_ep}...')
            run_on_trained(model, root_dir, run_fn, ep=ep)
    # If a specific number of epochs is provided:
    # 
    # Iterates through each epoch from 0 to n_ep-1
    # Prints progress information
    # Calls run_on_trained for each epoch
    
    else:
        for model_filename in sorted(root_dir.glob('*.pth')):
            print(f'running on checkpoint {model_filename}...')
            run_on_trained(model, root_dir, run_fn, model_filename=model_filename)

    # If no specific number of epochs is provided:
    # 
    # Finds all .pth files in the root directory
    # Sorts them (presumably by name, which would be by epoch if using the naming pattern)
    # Processes each checkpoint file in order
    
    print(f'done')

    # Prints a completion message when all checkpoints have been processed
    # 
    # These utility functions make it easy to:
    # 
    # Analyze a model at a specific point in its training history
    # Run the same analysis across multiple stages of training
    # Visualize or evaluate how the model's behavior changes over the course of training
    # 
    # They're particularly useful for post-training analysis, debugging, and creating visualizations of model evolution.

In [ ]:
def store_duration(time_analysis_dict, time_analysis_for_df_dict, analysis_name, duration,
                  timestamp_start_is_photo_analysis,
                  timestamp_end_is_photo_analysis):
    time_analysis_dict[analysis_name] = {}
    time_analysis_dict[analysis_name]['duration_str'] = f"Analysis took: {duration}"
    time_analysis_dict[analysis_name]['duration_seconds'] = total_seconds
    time_analysis_dict[analysis_name]['duration_seconds_str'] = f"Analysis took: {total_seconds:.2f} seconds"
    time_analysis_dict[analysis_name]['duration_minutes'] = total_seconds/60
    time_analysis_dict[analysis_name]['duration_minutes_str'] = f"Analysis took: {total_seconds/60:.2f} minutes"
    time_analysis_dict[analysis_name]['time_stamp_start'] = timestamp_start_is_photo_analysis
    time_analysis_dict[analysis_name]['time_stamp_end'] = timestamp_end_is_photo_analysis

    time_analysis_for_df_dict['analysis_name'].append(analysis_name)
    time_analysis_for_df_dict['time_stamp_start'].append(timestamp_start_is_photo_analysis)
    time_analysis_for_df_dict['duration_str'].append(f"Analysis took: {duration}")
    time_analysis_for_df_dict['duration_seconds'].append(total_seconds)
    time_analysis_for_df_dict['duration_seconds_str'].append(f"Analysis took: {total_seconds:.2f} seconds")
    time_analysis_for_df_dict['duration_minutes'].append(total_seconds/60)
    time_analysis_for_df_dict['duration_minutes_str'].append(f"Analysis took: {total_seconds/60:.2f} minutes")

    return time_analysis_dict, time_analysis_for_df_dic

### Prepare dictionaries to save meta data (hyperparameters) and results in:

In [ ]:
metadata_results = {}
autoencoder_params = {}


### Set Globals

In [ ]:
# Set globals (your existing code)
BATCH_SIZE = 32


device = torch.device("mps" if torch.backends.mps.is_built() else "cpu")
print("Using device:", device)


#N_SAMPLE = 4*32 # Used to calculate indices for visualisation.
N_SAMPLE = 10 # Used to calculate indices for visualisation.
N_VIS_SAMPLE = 2 # Used to calculate indices for visualisation.
CODE_SIZE = 120 # Number of features in the latent space.
NOISE_RATE = 0.1
MODEL_NAME = 'var_conv_ae'
SIDE_LENGTH = 320 

target_size = (SIDE_LENGTH, SIDE_LENGTH) # Size of images as the are fed into the autoencoder.

# ============================================================================
# CLUSTERING CONFIGURATION
# ============================================================================

CLUSTERING_METHOD = 'kmeans'  # Options: 'hierarchical', 'kmeans', 'gmm', 'dbscan'
N_FEATURES_FOR_CLUSTERING = 2  # Set to None for no reduction (use all CODE_SIZE dimensions)



### Set label name: 

In [ ]:
#label_name = 'buildings'
label_name = 'is_map'

### Record metadata: 

In [ ]:
# Record metadata: 
metadata_results['autoencoder_name'] = MODEL_NAME

autoencoder_params['code_size'] = CODE_SIZE
autoencoder_params['noise rate'] = NOISE_RATE
autoencoder_params['side length'] = SIDE_LENGTH



In [ ]:
metadata_results['analysis_type'] = 'clustering'
metadata_results['model_name'] = 'clustering pipeline'
metadata_results['python_script'] = 'img_to_pytorch_vae_nv.ipynb'
metadata_results['notes'] = 'variational convolutional autoencoder trained from scratch, training set = validation set, samples taken from training set.'
clustering_params = {'n_dimensions': N_FEATURES_FOR_CLUSTERING}


### Define file paths: 

In [ ]:

project_path = Path.cwd()
root_path = (project_path / '..' / 'data_folders').resolve()

# Define paths
#image_dir = root_path/'visual_genome_proc_data'  # Replace with your directory containing images
#data_path = root_path / 'visual_genome_data'

#image_dir = root_path/'combined_data'  # Replace with your directory containing images
#data_path = root_path / 'combined_data'

data_path = root_path / 'data'

image_dir = root_path / 'data_jpg'

### Get information about image size: 

In [ ]:
results = analyze_image_sizes(image_dir, extensions=('.jpg', '.jpeg', '.tif', '.tiff'))
results

### Load meta data about images: 

In [ ]:
meta_data = pd.read_csv(data_path/'labels_new.csv')
meta_data.head()

In [ ]:
meta_data.shape

In [ ]:
meta_data['file_paths'][4]

In [ ]:
#image_paths = load_image_paths(image_dir)
#image_paths[0:3]

In [ ]:
image_paths = list(meta_data.file_paths)
#labels = list(meta_data.mountains)
#labels = list(meta_data.buildings)
labels = list(meta_data[label_name])

### Get file sources: 

In [ ]:
file_sources = list(set(meta_data.file_source))
file_sources

### Define Basic Autoencoder Class

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_size, code_size):
        self.input_size = list(input_size)  # shape of data sample
        self.flat_data_size = np.prod(self.input_size)
        self.hidden_size = 128

        self.code_size = code_size  # code size

        super(AutoEncoder, self).__init__()
        #Creates an autoencoder neural network that inherits from PyTorch's nn.Module.
        #Takes two parameters:
        #
        #input_size: The shape of input data (e.g., [1, 28, 28] for MNIST)
        #code_size: The dimension of the encoded representation (bottleneck)
        #
        #
        #Calculates the flattened input size by multiplying all dimensions.
        #Sets an intermediate hidden layer size of 128 neurons.
        #Calls the parent class initializer.

        
        self.encoder = nn.Sequential(
            nn.Flatten(),

            nn.Linear(self.flat_data_size, self.hidden_size),
            nn.ReLU(),

            nn.Linear(self.hidden_size, self.code_size),
            nn.Sigmoid(),
        )
        #Defines the encoder network as a sequence of operations:
            #
            #Flattens the input (e.g., converts a 2D image to 1D)
            #Linear layer mapping from input size to hidden size
            #ReLU activation
            #Linear layer mapping from hidden size to code size
            #Sigmoid activation (constrains the encoded values to [0, 1])
        
        self.decoder = nn.Sequential(
            nn.Linear(self.code_size, self.hidden_size),
            nn.ReLU(),

            nn.Linear(self.hidden_size, self.flat_data_size),
            nn.Tanh(),  # Think: why tanh?

            nn.Unflatten(1, self.input_size),
        )
        #Defines the decoder network:
            #Linear layer from code size to hidden size
            #ReLU activation
            #Linear layer from hidden size back to the flattened input size
            #Tanh activation (outputs values in [-1, 1], matching the normalized input range)
            #Unflattens the output back to the original input shape

#Regarding "why tanh?": Tanh is used because the input images were normalized to approximately [-0.5, 0.5] 
    #range (using m=0.5, s=1.0). Tanh outputs values in the range [-1, 1], 
    #which after scaling by 1.1 in the decode method closely matches the input data range.

    def forward(self, x, return_z=False):
        encoded = self.encode(x)
        decoded = self.decode(encoded)
        return (decoded, encoded) if return_z else decoded
    # The forward pass:
        #Encodes the input
        #Decodes the encoded representation
        #If return_z=True, returns both the reconstruction and the encoded values
        #Otherwise, just returns the reconstruction
        

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)*1.1
# Helper methods to encode and decode separately
# Note the multiplication by 1.1 in the decode method, 
    # which slightly amplifies the output range to better match the input data distribution

        

    def get_n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    # Utility method to count the total number of trainable parameters in the model


# def eval_on_samples(ae_model, epoch, samples):
#     # this is called on end of each training epoch
#     xns = samples['images_noisy']
#     xns = torch.tensor(xns, dtype=torch.float32).to(device)
#     #labels = samples['labels']


def eval_on_samples(ae_model, epoch, samples):
    """
    Evaluate autoencoder on sample images.
    
    For VAE: returns reconstruction + latent distribution parameters (z_mean, z_logvar)
    For regular AE: returns reconstruction + latent code (z)
    
    Args:
        ae_model: The autoencoder model (VAE or regular AE)
        epoch: Current epoch number
        samples: Dictionary containing sample images
        
    Returns:
        Dictionary with 'z', 'y', and 'epoch' keys
        For VAE: z is a list [z_mean, z_logvar]
    """
    # Extract noisy images from samples
    xns = samples['images_noisy']
    xns = torch.tensor(xns, dtype=torch.float32).to(device)
    
    # Evaluation mode - no gradient calculation needed
    ae_model.eval()
    with torch.no_grad():
        # Forward pass with return_z=True to get latent representations
        yz = ae_model(xns, return_z=True)
        
        # Move results to CPU and convert to numpy
        yz = [el.detach().cpu().numpy() for el in yz]
        
        # Extract components
        y = yz[0]           # Reconstructed images
        z = yz[1:]          # Latent representations
        
        # For VAE: z is a list [z_mean, z_logvar] (2 components)
        # For regular AE: z is a list [z] (1 component)
    
    # Return results as dictionary
    sample_res = {'z': z, 'y': y, 'epoch': epoch}
    
    return sample_res

# Function to evaluate the autoencoder on sample data after each epoch
# Takes the model, current epoch number, and samples dictionary
# Extracts noisy images from the samples and converts them to a PyTorch tensor on the target device
# The labels are extracted but commented out (not used)

    with torch.no_grad():
        yz = ae_model(xns, return_z=True)
        yz = [el.detach().cpu().numpy() for el in yz]

        y = yz[0]
        z = yz[1:]
    # Uses torch.no_grad() to disable gradient calculation (for efficiency during evaluation)
    # Gets both reconstructions and encodings (i.e. latent space!) by calling the model with return_z=True
    # Converts all outputs to NumPy arrays
    # Separates the reconstructions y and encodings z

    res = {'z': z, 'y': y, 'epoch': epoch}
    return res

# Creates and returns a dictionary containing:

# z: The encoded representations
# y: The reconstructed images
# epoch: The current epoch number
# 

# This evaluation function captures the model's performance at each epoch, allowing for tracking reconstruction quality and analyzing the learned representations over time.

### For non-validated training use all images for training, i.e. use all image paths:


In [ ]:
train_paths = image_paths
train_labels = labels

### Load Data

In [ ]:
# Step 1: Load image paths
# already done

# Step 2: Create your labels list (matching image_paths order)
# already done

# Step 3: Create dataset (equivalent to train_dataset)
train_dataset = create_image_dataset(
    image_paths=train_paths,
    labels=train_labels,
    target_size=target_size,
    convert_grayscale=True,
    apply_aging=False,
    mean=0.5,
    std=1.0
)

In [ ]:
type(train_dataset)

In [ ]:


# Step 4: Create DataLoader with your original collate function
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_ae_dataset,  # Your original function using globals
    drop_last=True
)


In [ ]:
type(train_loader)

### For non-validated training set valid_dataset = train_dataset:

In [ ]:
valid_dataset = train_dataset
valid_loader = train_loader

### Check data properties:

In [ ]:
# Diagnostic code for MNIST properties
#train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform, target_transform=lable_transform)

print("=== MNIST Dataset Properties ===")
print(f"Type: {type(train_dataset)}")
print(f"Length: {len(train_dataset)}")

# Check all attributes
print(f"\nAttributes: {[attr for attr in dir(train_dataset) if not attr.startswith('_')]}")

# Check key properties
if hasattr(train_dataset, 'data'):
    print(f"data type: {type(train_dataset.data)}")
    print(f"data shape: {train_dataset.data.shape}")
    print(f"data dtype: {train_dataset.data.dtype}")

if hasattr(train_dataset, 'image_paths'):
    print(f"image_paths type: {type(train_dataset.image_paths)}")
    print(f"image_paths length: {len(train_dataset.image_paths)}")
    print(f"image_pathsdtype: {type(train_dataset.image_paths[0])}")

if hasattr(train_dataset, 'targets'):
    print(f"targets type: {type(train_dataset.targets)}")
    print(f"targets shape: {train_dataset.targets.shape if hasattr(train_dataset.targets, 'shape') else len(train_dataset.targets)}")

# Test __getitem__
sample = train_dataset[0]
print(f"\n__getitem__ returns: {type(sample)}")
print(f"Sample[0] type: {type(sample[0])}, shape: {sample[0].shape}")
print(f"Sample[1] type: {type(sample[1])}")

In [ ]:
print(type(train_dataset.image_paths))

In [ ]:
train_dataset.image_paths[0:3]

In [ ]:
type(train_dataset.image_paths[0])

### Store file paths of training and validation images:

In [ ]:
train_data_file_paths = pd.DataFrame({'filepaths': train_dataset.image_paths})
val_data_file_paths = pd.DataFrame({'filepaths': valid_dataset.image_paths})

In [ ]:
train_dataset.image_paths[0]

In [ ]:
train_dataset.image_paths[2]

In [ ]:
# Diagnostic code for MNIST properties
#valid_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform, target_transform=lable_transform)

print("=== MNIST Dataset Properties ===")

print(f"Type: {type(valid_dataset)}")
print(f"Length: {len(valid_dataset)}")

# Check all attributes
print(f"\nAttributes: {[attr for attr in dir(valid_dataset) if not attr.startswith('_')]}")

# Check key properties
if hasattr(valid_dataset, 'data'):
    print(f"data type: {type(valid_dataset.data)}")
    print(f"data shape: {valid_dataset.data.shape}")
    print(f"data dtype: {valid_dataset.data.dtype}")

if hasattr(valid_dataset, 'image_paths'):
    print(f"image_paths type: {type(valid_dataset.image_paths)}")
    print(f"image_paths length: {len(valid_dataset.image_paths)}")
    print(f"image_pathsdtype: {type(valid_dataset.image_paths[0])}")

if hasattr(valid_dataset, 'targets'):
    print(f"targets type: {type(valid_dataset.targets)}")
    print(f"targets shape: {valid_dataset.targets.shape if hasattr(valid_dataset.targets, 'shape') else len(valid_dataset.targets)}")

# Test __getitem__
sample = valid_dataset[0]
print(f"\n__getitem__ returns: {type(sample)}")
print(f"Sample[0] type: {type(sample[0])}, shape: {sample[0].shape}")
print(f"Sample[1] type: {type(sample[1])}")

In [ ]:
# Diagnostic code for train_loader object
print("=== TRAIN_LOADER Properties ===")
print(f"Type: {type(train_loader)}")
print(f"Length (number of batches): {len(train_loader)}")

# Check all attributes
print(f"\nAttributes: {[attr for attr in dir(train_loader) if not attr.startswith('_')]}")

# Check key properties
print(f"batch_size: {train_loader.batch_size}")
#print(f"shuffle: {train_loader.shuffle}")
print(f"drop_last: {train_loader.drop_last}")
print(f"collate_fn: {train_loader.collate_fn}")
print(f"dataset type: {type(train_loader.dataset)}")
print(f"dataset length: {len(train_loader.dataset)}")

# Test getting one batch
print(f"\n=== BATCH SAMPLE ===")
for batch in train_loader:
   print(f"Batch type: {type(batch)}")
   print(f"Batch length: {len(batch)}")
   
   # Unpack the batch (based on your collate function returning 3 items)
   noisy_img, clean_img, labels = batch
   
   print(f"\nnoisy_img type: {type(noisy_img)}")
   print(f"noisy_img shape: {noisy_img.shape}")
   print(f"noisy_img dtype: {noisy_img.dtype}")
   print(f"noisy_img device: {noisy_img.device}")
   
   print(f"\nclean_img type: {type(clean_img)}")
   print(f"clean_img shape: {clean_img.shape}")
   print(f"clean_img dtype: {clean_img.dtype}")
   print(f"clean_img device: {clean_img.device}")
   
   print(f"\nlabels type: {type(labels)}")
   print(f"labels shape: {labels.shape}")
   print(f"labels dtype: {labels.dtype}")
   print(f"labels device: {labels.device}")
   
   break  # Only test first batch

### Define Variational Convolutional Autoencoder (inherits some properties from basic autoencoder)

In [ ]:
# Calculate final spatial size after convolutional layers
# For 320×320 input with 6 stride-2 layers: 320 ÷ (2^6) = 320 ÷ 64 = 5×5
final_spatial_size = SIDE_LENGTH // (2**6)
print(f"Final spatial size after convolutions: {final_spatial_size}×{final_spatial_size}")

# This will be used to calculate hidden_size for the VAE
# hidden_size = 64 channels × 5 × 5 = 1600

In [ ]:
# Variational Convolutional Autoencoder (VAE) for 320×320 images
# Adapted from tutorial notebook for larger image size
# Key difference from regular autoencoder: encodes distributions (mean + log variance)
# instead of deterministic latent codes. This creates better structured latent space
# for clustering and generation.

class VariationalConvolutionalAutoencoder(AutoEncoder):
    """
    Variational Convolutional Autoencoder adapted for 320×320 images.
    
    Architecture:
    - Encoder: 6 stride-2 conv layers (320→160→80→40→20→10→5) + fully connected
    - Outputs: mean and log_variance for latent distribution
    - Decoder: Fully connected + 6 transpose conv layers (5→10→20→40→80→160→320)
    
    Inherits from AutoEncoder base class but implements variational approach.
    """
    
    def __init__(self, input_size, code_size):
        super(VariationalConvolutionalAutoencoder, self).__init__(input_size, code_size)

        self.input_size = list(input_size)  # Shape of data sample (e.g., [1, 320, 320])
        self.npix = np.prod(self.input_size)  # Total number of pixels (used for loss normalization)

        # For 320×320 input with 6 stride-2 layers: final spatial size is 5×5
        # With 64 output channels: 64 × 5 × 5 = 1600 values before fully connected layers
        self.hidden_size = 64 * 5 * 5  # = 1600
        
        self.code_size = code_size  # Latent code dimension
        
        # ENCODER: Compresses 320×320 image to latent distribution parameters
        # Note: Final layer outputs 2×code_size values (mean + log_variance)
        self.encoder = nn.Sequential(
            # Convolutional layers with progressive downsampling
            nn.Conv2d(1, 8, 3, padding=1, stride=1), nn.LeakyReLU(negative_slope=0.3),      # 320×320×8
            nn.Conv2d(8, 8, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),      # 160×160×8
            nn.Conv2d(8, 16, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),     # 80×80×16
            nn.Conv2d(16, 16, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),    # 40×40×16
            nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),    # 20×20×32
            nn.Conv2d(32, 32, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),    # 10×10×32
            nn.Conv2d(32, 64, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),    # 5×5×64
            
            nn.Flatten(),  # Flatten to: 64×5×5 = 1600 values
            
            # Fully connected layers to compress to latent space
            nn.Linear(1600, 200), nn.LeakyReLU(negative_slope=0.3),  # 1600 → 200
            
            # Output layer: produces BOTH mean and log_variance (hence 2×code_size)
            # No activation function - we want values in all real numbers
            nn.Linear(200, self.code_size * 2),  # 200 → (code_size×2)
        )
        
        # DECODER: Reconstructs 320×320 image from latent code
        self.decoder = nn.Sequential(
            # Fully connected layers to expand from latent space
            nn.Linear(self.code_size, 200), nn.LeakyReLU(negative_slope=0.3),  # code_size → 200
            nn.Linear(200, 1600), nn.LeakyReLU(negative_slope=0.3),            # 200 → 1600
            
            # Reshape to spatial feature maps
            nn.Unflatten(1, (64, 5, 5)),  # 1600 → 64×5×5
            
            # Transposed convolutions for progressive upsampling
            nn.ConvTranspose2d(64, 32, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),  # 5→10
            nn.ConvTranspose2d(32, 32, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),  # 10→20
            nn.ConvTranspose2d(32, 16, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),  # 20→40
            nn.ConvTranspose2d(16, 16, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),  # 40→80
            nn.ConvTranspose2d(16, 8, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),   # 80→160
            nn.ConvTranspose2d(8, 8, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),    # 160→320
            
            # Final layer to produce single-channel output
            nn.Conv2d(8, 1, 3, padding=1, stride=1), nn.Tanh(),  # 320×320×1, output in [-1, 1]
        )

    def sample(self, eps=None):
        """
        Generate random samples from the latent space.
        
        Args:
            eps: Optional noise tensor. If None, generates 100 random samples.
        
        Returns:
            Generated images sampled from standard normal distribution in latent space.
        """
        if eps is None:
            # Sample from standard normal distribution N(0, 1)
            eps = torch.randn((100, self.code_size))
        return self.decode(eps)
    
    def encode(self, x):
        """
        Encode input images to latent distribution parameters.
        
        Args:
            x: Input images tensor
            
        Returns:
            z_mean: Mean of latent distribution
            z_logvar: Log variance of latent distribution
            
        Note: This is different from regular autoencoder which returns single latent code.
        VAE encodes as distributions to create more structured latent space.
        """
        z = self.encoder(x)
        # Split encoder output into mean and log variance
        # split_size_or_sections=2 means split into 2 equal parts along dim=1
        z_mean, z_logvar = torch.split(z, self.code_size, dim=1)
        return z_mean, z_logvar

    def reparameterize(self, z_mean, z_logvar):
        """
        Reparameterization trick: sample from N(mean, std) in a differentiable way.
        
        Instead of z ~ N(mean, std) which is not differentiable,
        we use: z = mean + std × ε, where ε ~ N(0, 1)
        
        This allows gradients to flow through the sampling operation during backprop.
        
        Args:
            z_mean: Mean of the distribution
            z_logvar: Log variance of the distribution
            
        Returns:
            z: Sampled latent code
        """
        eps = torch.randn_like(z_mean)  # Sample noise from standard normal
        z_std = torch.exp(z_logvar * 0.5)  # Convert log variance to std deviation
        return eps * z_std + z_mean  # Reparameterization: z = mean + std*noise

    def decode(self, z):
        """
        Decode latent code to reconstructed image.
        
        Args:
            z: Latent code tensor
            
        Returns:
            Reconstructed image (no cropping needed - dimensions match exactly)
        """
        reconstruction = self.decoder(z)
        # Note: Original tutorial had cropping (2:-2) for 28×28 images
        # For 320×320 with proper architecture, no cropping needed
        return reconstruction

    def forward(self, x, return_z=False):
        """
        Complete forward pass through the VAE.
        
        Args:
            x: Input images
            return_z: If True, also return latent distribution parameters
            
        Returns:
            If return_z=False: reconstruction only
            If return_z=True: (reconstruction, z_mean, z_logvar)
        """
        z_mean, z_logvar = self.encode(x)
        z = self.reparameterize(z_mean, z_logvar)
        reconstruction = self.decode(z)
        
        if return_z:
            return reconstruction, z_mean, z_logvar
        else:
            return reconstruction

    def forward_and_KL_loss(self, x, y):
        """
        Forward pass with KL divergence loss calculation.
        
        VAE loss = Reconstruction loss + KL divergence loss
        
        KL divergence measures how much the learned latent distribution
        differs from a standard normal distribution N(0,1).
        This regularizes the latent space to be well-structured.
        
        Args:
            x: Input images (noisy)
            y: Target images (clean) - not used in this method but kept for compatibility
            
        Returns:
            reconstruction: Reconstructed images
            loss_z_kl: KL divergence loss term
        """
        reconstruction, z_mean, z_logvar = self(x, return_z=True)

        # KL divergence from N(z_mean, exp(z_logvar/2)) to N(0, 1)
        # Formula: 0.5 * sum(exp(logvar) + mean^2 - 1 - logvar)
        loss_z_kl = 0.5 * torch.sum(
            torch.exp(z_logvar) + torch.square(z_mean) - 1.0 - z_logvar, 
            dim=1
        )
        
        # Normalize by number of pixels (for proper scaling with reconstruction loss)
        loss_z_kl = torch.mean(loss_z_kl) / self.npix

        return reconstruction, loss_z_kl
        

### Instantiate the Variational Convolutional Autoencoder

In [ ]:
# Instantiate the Variational Convolutional Autoencoder
in_size = target_size  # Should be (1, 320, 320)
model = VariationalConvolutionalAutoencoder(input_size=in_size, code_size=CODE_SIZE).to(device)

print(f"✓ VAE model created and moved to device: {device}")
print(f"Input size: {in_size}")
print(f"Code size: {CODE_SIZE}")
print(f"Hidden size: {model.hidden_size}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")



### Get samples from training or validation dataset (using your custom function):

In [ ]:
# Get samples from training or validation dataset (using your custom function)
#samples = get_samples_from_dataset(valid_dataset)
samples = get_samples_from_dataset_all(train_dataset)
print(f"\n✓ Samples extracted from training dataset")
print(f"Sample image_paths shape: {samples['image_paths'].shape}")


### Record metadata: 

In [ ]:
# Record metadata
metadata_results['autoencoder_implementation'] = 'Custom PyTorch VariationalConvolutionalAutoencoder'
autoencoder_params['in_size'] = in_size # in_size = target_size , sorry for the redundant name change.

# Display sample information
print(f"\nSample structure:")
print(f"  Type: {type(samples)}")
print(f"  Keys: {list(samples.keys())}")
print(f"  images_noisy shape: {samples['images_noisy'].shape}")
print(f"  images shape: {samples['images'].shape}")
print(f"  labels shape: {samples['labels'].shape}")
print(f"  image_paths shape: {samples['image_paths'].shape}")
print(f"  single_el_idx shape: {samples['single_el_idx'].shape}")

### Test the VAE with sample data to verify everything works:

In [ ]:
# Test the VAE with sample data to verify everything works
xns = torch.tensor(samples['images_noisy']).to(device)
print(f"Input tensor shape: {xns.shape}")

# Test encode method (VAE returns mean and log_variance)
with torch.no_grad():
    z_mean, z_logvar = model.encode(xns)
    print(f"\n✓ Encoding successful!")
    print(f"  z_mean shape: {z_mean.shape}")
    print(f"  z_logvar shape: {z_logvar.shape}")

# Test forward pass
with torch.no_grad():
    ys = model(xns)
    print(f"\n✓ Forward pass successful!")
    print(f"  Output shape: {ys.shape}")

# Test forward_and_KL_loss method (used during training)
with torch.no_grad():
    reconstruction, kl_loss = model.forward_and_KL_loss(xns, xns)
    print(f"\n✓ forward_and_KL_loss successful!")
    print(f"  Reconstruction shape: {reconstruction.shape}")
    print(f"  KL loss value: {kl_loss.item():.6f}")

### Initialize timing and tracking dictionaries:

In [ ]:
# Initialize timing and tracking dictionaries
time_analyses = {}
time_analyses_clustering_pipeline = {}
time_analyses_clustering_pipeline['analysis_name'] = []
time_analyses_clustering_pipeline['time_stamp_start'] = []
time_analyses_clustering_pipeline['duration_str'] = []
time_analyses_clustering_pipeline['duration_seconds'] = []
time_analyses_clustering_pipeline['duration_seconds_str'] = []
time_analyses_clustering_pipeline['duration_minutes'] = []
time_analyses_clustering_pipeline['duration_minutes_str'] = []

timestamp_start_workflow = pd.Timestamp.now()
timestamp_start_clustering_pipeline = pd.Timestamp.now()
metadata_results['run_timestamp'] = timestamp_start_clustering_pipeline

print(f"Training workflow started at: {timestamp_start_workflow}")

### Train the Variational Autoencoder:

In [ ]:
# Train the Variational Autoencoder
# VAE uses combined loss: Reconstruction Loss + KL Divergence Loss
# KL divergence encourages the latent space to follow a standard normal distribution,
# creating a more structured and meaningful latent representation

N_EPOCHS = 80
LR = 0.0009

# Create model directory
timestamp_id = timestamp_start_clustering_pipeline.strftime('%Y%m%d_%H%M%S')
#MODEL_NAME = f'cdae_model_{timestamp_id}'
MODEL_NAME_TIMESTAMP = MODEL_NAME + '_' + timestamp_id
#model_root = pl.Path(MODEL_NAME)
model_root = data_path / MODEL_NAME_TIMESTAMP
model_root.mkdir(exist_ok=True)
print(f"Model checkpoints will be saved to: {model_root}")

# Initialize optimizer
optimizer = optim.Adam(model.parameters(), lr=LR)

# Define reconstruction loss function (MSE)
reconstruction_loss_fn = nn.MSELoss()

# Initialize training history
history = {
    'loss': [],           # Total loss (reconstruction + KL)
    'recon_loss': [],     # Reconstruction loss only
    'kl_loss': [],        # KL divergence loss only
    'val_loss': [],       # Total validation loss
    'val_recon_loss': [], # Validation reconstruction loss
    'val_kl_loss': [],    # Validation KL divergence loss
    'epoch': []
}
sample_history = {}

print(f"\nStarting VAE training:")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Learning rate: {LR}")
print(f"  Optimizer: Adam")
print(f"  Reconstruction loss: MSE")
print(f"  Device: {device}")

# Create progress bar
pbar = tqdm.tqdm(range(N_EPOCHS), postfix=f'epoch 0/{N_EPOCHS}')

for epoch_idx in pbar:
    # ========== TRAINING PHASE ==========
    epoch_loss = 0
    epoch_recon_loss = 0
    epoch_kl_loss = 0
    model.train()
    
    for batch_idx, (noisy_data, data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        
        # VAE forward pass with KL loss calculation
        # model.forward_and_KL_loss returns (reconstruction, kl_loss)
        reconstruction, kl_loss = model.forward_and_KL_loss(noisy_data, data)
        
        # Calculate reconstruction loss (MSE between reconstruction and clean data)
        recon_loss = reconstruction_loss_fn(reconstruction, data)
        
        # Total VAE loss = Reconstruction loss + KL divergence loss
        total_loss = recon_loss + kl_loss
        
        # Backpropagation and optimization
        total_loss.backward()
        optimizer.step()
        
        # Accumulate losses for epoch statistics
        epoch_loss += total_loss.detach().cpu().item()
        epoch_recon_loss += recon_loss.detach().cpu().item()
        epoch_kl_loss += kl_loss.detach().cpu().item()
    
    # Calculate average losses for the epoch
    epoch_loss /= len(train_loader)
    epoch_recon_loss /= len(train_loader)
    epoch_kl_loss /= len(train_loader)
    
    # Record training losses
    history['loss'].append(epoch_loss)
    history['recon_loss'].append(epoch_recon_loss)
    history['kl_loss'].append(epoch_kl_loss)
    history['epoch'].append(epoch_idx)
    
    # ========== VALIDATION PHASE ==========
    model.eval()
    with torch.no_grad():
        val_loss = 0
        val_recon_loss = 0
        val_kl_loss = 0
        
        for batch_idx, (noisy_data, data, target) in enumerate(valid_loader):
            # Forward pass with KL loss
            reconstruction, kl_loss = model.forward_and_KL_loss(noisy_data, data)
            
            # Calculate reconstruction loss
            recon_loss = reconstruction_loss_fn(reconstruction, data)
            
            # Total loss
            total_loss = recon_loss + kl_loss
            
            # Accumulate validation losses
            val_loss += total_loss.detach().cpu().item()
            val_recon_loss += recon_loss.detach().cpu().item()
            val_kl_loss += kl_loss.detach().cpu().item()
        
        # Calculate average validation losses
        val_loss /= len(valid_loader)
        val_recon_loss /= len(valid_loader)
        val_kl_loss /= len(valid_loader)
        
        # Record validation losses
        history['val_loss'].append(val_loss)
        history['val_recon_loss'].append(val_recon_loss)
        history['val_kl_loss'].append(val_kl_loss)
    
    # Update progress bar with all loss metrics
    pbar.set_postfix({
        'epoch': f'{epoch_idx+1}/{N_EPOCHS}',
        'loss': f'{epoch_loss:.4f}',
        'recon': f'{epoch_recon_loss:.4f}',
        'kl': f'{epoch_kl_loss:.4f}',
        'val_loss': f'{val_loss:.4f}'
    })
    
    # ========== SAMPLE EVALUATION ==========
    # Evaluate on sample images to track reconstruction quality
    sample_res = eval_on_samples(model, epoch_idx, samples=samples)
    sample_history[epoch_idx] = sample_res
    
    # ========== CHECKPOINT SAVING ==========
    # Save model checkpoint
    torch.save({
        'epoch': epoch_idx,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss,
        'recon_loss': epoch_recon_loss,
        'kl_loss': epoch_kl_loss,
        'val_loss': val_loss,
        'history': history
    }, model_root / f'model_{epoch_idx:03d}.pth')

# Record final training parameters
autoencoder_params['number_epochs'] = N_EPOCHS
autoencoder_params['learning_rate'] = LR
autoencoder_params['loss_type'] = 'VAE (Reconstruction + KL Divergence)'

print(f"\n✓ Training complete!")
print(f"Final training loss: {epoch_loss:.4f} (recon: {epoch_recon_loss:.4f}, kl: {epoch_kl_loss:.4f})")
print(f"Final validation loss: {val_loss:.4f} (recon: {val_recon_loss:.4f}, kl: {val_kl_loss:.4f})")

In [ ]:
# Plot training and validation losses
plot_hist(history)

### Store metadata: 

In [ ]:
# Store autoencoder_params:

metadata_results['autoencoder_params'] = autoencoder_params

### Save train and val file paths:

In [ ]:
train_data_file_paths_filename = 'train_data_file_paths_' + timestamp_id + '.csv'
train_data_file_paths_filepath = data_path / train_data_file_paths_filename
train_data_file_paths.to_csv(train_data_file_paths_filepath, index=False)


In [ ]:
val_data_file_paths_filename = 'val_data_file_paths_' + timestamp_id + '.csv'
val_data_file_paths_filepath = data_path / val_data_file_paths_filename
val_data_file_paths.to_csv(val_data_file_paths_filepath, index=False)


### Examine the structure of sample history:

In [ ]:
# Examine the structure of sample history
print("Sample history structure:")
print(f"  Keys for epoch 5: {sample_history[5].keys()}")
print(f"  Reconstruction shape: {sample_history[5]['y'].shape}")
print(f"  Latent z components:")

# VAE stores TWO latent components: z_mean and z_logvar
# sample_history[epoch]['z'] should contain [z_mean, z_logvar]
for i, z_component in enumerate(sample_history[5]['z']):
    print(f"    z[{i}] shape: {z_component.shape}")

In [ ]:
samples.keys()

In [ ]:
print(type(samples.keys()))
print(type(samples['images_noisy']))
print((samples['images_noisy'].shape))
print(type(samples['images']))
print((samples['images'].shape))
print(type(samples['labels']))
print((samples['labels'].shape))
print(type(samples['image_paths']))
print((samples['image_paths'].shape))
print(type(samples['single_el_idx']))
print((samples['single_el_idx'].shape))

In [ ]:
def subsample(samples, n, random_seed=42):
    """Extract a random subsample from a samples dictionary, preserving its structure."""
    np.random.seed(random_seed)
    indices = np.random.choice(len(samples['images']), n, replace=False)
    
    return {
        'images_noisy': samples['images_noisy'][indices],
        'images': samples['images'][indices],
        'labels': samples['labels'][indices],
        'image_paths': samples['image_paths'][indices],
        'single_el_idx': np.arange(n)  # reset indices for the new subsample
    }

# Usage:
samples_sub = subsample(samples, n=10)

### Visualize how reconstructions improve over epochs:

In [ ]:
# Visualize how reconstructions improve over epochs
plot_samples(sample_history, samples=samples_sub, epoch_stride=10, fig_scale=1)

In [ ]:
%%capture

# Extract sample images for animation
single_el_idx = samples['single_el_idx']
images_noisy = samples['images_noisy'][single_el_idx, 0]
images = samples['images'][single_el_idx, 0]

# Create list of image arrays for each epoch
smpl_ims = []
for epoch_idx, hist_el in sample_history.items():
    samples_arr = [images_noisy, hist_el['y'][single_el_idx, 0], images]
    smpl_ims.append(samples_arr)

ny, nx = len(smpl_ims[0]), len(smpl_ims[0][0])

# Create animation
plt.rcParams["animation.html"] = "jshtml"
s = 1
fig = plt.figure(figsize=(s*nx, s*ny))

m = mosaic(smpl_ims[0])
ttl = plt.title(f'after epoch {int(0)}')
imsh = plt.imshow(m, cmap='gray', vmin=-0.5, vmax=0.5)

def animate(i):
    m = mosaic(smpl_ims[i])
    imsh.set_data(m)
    ttl.set_text(f'after epoch {i}')
    return imsh

ani = animation.FuncAnimation(fig, animate, frames=len(smpl_ims))

In [ ]:
# Display the reconstruction animation
ani

In [ ]:
%%capture

# Prepare latent space visualization
plt.rcParams["animation.html"] = "jshtml"
fig = plt.figure(figsize=(8, 8))

labels = samples['labels']
epochs = sorted(sample_history.keys())

# IMPORTANT: For VAE, use z_mean (first component) for visualization
# VAE's sample_history['z'] contains [z_mean, z_logvar]
# We use z_mean because it represents the actual latent code location
z_res = [sample_history[ep]['z'][0] for ep in epochs]  # z[0] is z_mean

# Create initial scatter plot using two dimensions of latent space
scat = plt.scatter(z_res[0][:, 0], z_res[0][:, 1], c=labels, cmap=cm.rainbow)

plt.xlim(-60.1, 60.1)
plt.ylim(-60.1, 60.1)

ax = plt.gca()
legend1 = ax.legend(*scat.legend_elements(), title="classes")
ax.add_artist(legend1)
ax.set_aspect('equal')
ttl = plt.title(f'after epoch {0}')

def animate(i):
    z = z_res[i]
    scat.set_offsets(z[:, :2])  # Use first 2 dimensions
    ttl.set_text(f'after epoch {i}')
    return scat

ani = animation.FuncAnimation(fig, animate, frames=len(z_res))

In [ ]:
# Display latent space animation
ani

### Perform clustering on latent space:

In [ ]:
# ============================================================================
# STEP 1: Extract final epoch latent codes
# ============================================================================

from sklearn.decomposition import PCA

final_epoch = max(sample_history.keys())
z_mean = sample_history[final_epoch]['z'][0]

print(f"Extracted latent codes from epoch {final_epoch}")
print(f"Shape: {z_mean.shape}")

# ============================================================================
# STEP 2: Optional dimensionality reduction
# ============================================================================

if N_FEATURES_FOR_CLUSTERING is None or N_FEATURES_FOR_CLUSTERING >= CODE_SIZE:
    z_reduced = z_mean  # No reduction
    print(f"Using full latent space: {z_reduced.shape[1]} dimensions")

    metadata_results['dim_reduction_name'] = None,
    metadata_results['dim_reduction_implementation'] = None
    metadata_results['dim_reduction_params'] = None
    
else:
    pca = PCA(n_components=N_FEATURES_FOR_CLUSTERING)
    z_reduced = pca.fit_transform(z_mean)
    print(f"PCA reduced to {z_reduced.shape[1]} dimensions")
    print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

    metadata_results['dim_reduction_name'] = 'PCA'
    metadata_results['dim_reduction_implementation'] = 'sklearn.decomposition'
    metadata_results['dim_reduction_params'] ={'n_dimensions': N_FEATURES_FOR_CLUSTERING}

In [ ]:
# ============================================================================
# STEP 3: Apply clustering method & show diagnostic plots
# ============================================================================

import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import numpy as np

if CLUSTERING_METHOD == 'hierarchical':
    print("="*80)
    print("HIERARCHICAL CLUSTERING")
    print("="*80)
    
    # Build linkage matrix
    linkage_matrix = linkage(z_reduced, method='ward')
    
    # Show dendrogram to choose number of clusters
    plt.figure(figsize=(15, 7))
    dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
    plt.title('Hierarchical Clustering Dendrogram', fontsize=16)
    plt.xlabel('Sample Index (or Cluster Size)')
    plt.ylabel('Distance')
    plt.show()
    
    print(f"\n→ Adjust N_CLUSTERS based on dendrogram, then run next cell")

elif CLUSTERING_METHOD == 'kmeans':
    print("="*80)
    print("K-MEANS CLUSTERING")
    print("="*80)
    
    # Elbow method: plot inertia for different k
    inertias = []
    K_range = range(2, 16)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        kmeans.fit(z_reduced)
        inertias.append(kmeans.inertia_)
    
    plt.figure(figsize=(10, 5))
    plt.plot(K_range, inertias, 'bo-')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Inertia (within-cluster sum of squares)')
    plt.title('Elbow Method for Optimal k')
    plt.grid(True)
    plt.show()
    
    # Silhouette scores
    silhouette_scores = []
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(z_reduced)
        score = silhouette_score(z_reduced, labels)
        silhouette_scores.append(score)
    
    plt.figure(figsize=(10, 5))
    plt.plot(K_range, silhouette_scores, 'go-')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Silhouette Score')
    plt.title('Silhouette Analysis for Optimal k')
    plt.grid(True)
    plt.show()
    
    print(f"\n→ Adjust N_CLUSTERS based on elbow/silhouette, then run next cell")

elif CLUSTERING_METHOD == 'gmm':
    print("="*80)
    print("GAUSSIAN MIXTURE MODEL (GMM)")
    print("="*80)
    
    # BIC/AIC scores for model selection
    bic_scores = []
    aic_scores = []
    K_range = range(2, 16)
    
    for k in K_range:
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
        gmm.fit(z_reduced)
        bic_scores.append(gmm.bic(z_reduced))
        aic_scores.append(gmm.aic(z_reduced))
    
    plt.figure(figsize=(10, 5))
    plt.plot(K_range, bic_scores, 'ro-', label='BIC')
    plt.plot(K_range, aic_scores, 'bo-', label='AIC')
    plt.xlabel('Number of components (k)')
    plt.ylabel('Information Criterion')
    plt.title('GMM Model Selection (lower is better)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print(f"\n→ Choose N_CLUSTERS where BIC/AIC is lowest, then run next cell")

elif CLUSTERING_METHOD == 'dbscan':
    print("="*80)
    print("DBSCAN (Density-Based Clustering)")
    print("="*80)
    print("Note: DBSCAN automatically determines number of clusters")
    print("You need to set eps (neighborhood radius) and min_samples")
    print("\n→ Run next cell with default parameters, adjust if needed")

else:
    raise ValueError(f"Unknown clustering method: {CLUSTERING_METHOD}")

In [ ]:
# ============================================================================
# DEFINE NUMBER OF CLUSTERS (after inspecting diagnostic plots above)
# ============================================================================

N_CLUSTERS = 2  # Adjust based on dendrogram/elbow/silhouette plots

In [ ]:
# ============================================================================
# STEP 4: Apply clustering and get labels
# ============================================================================

if CLUSTERING_METHOD == 'hierarchical':
    cluster_labels = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')
    # Convert to 0-indexed
    cluster_labels = cluster_labels - 1
    print(f"✓ Hierarchical clustering complete: {N_CLUSTERS} clusters")
    metadata_results['clustering_implementation'] =  'scipy.cluster.hierarchy'

elif CLUSTERING_METHOD == 'kmeans':
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(z_reduced)
    print(f"✓ K-means clustering complete: {N_CLUSTERS} clusters")
    metadata_results['clustering_implementation'] = 'sklearn.cluster'

elif CLUSTERING_METHOD == 'gmm':
    gmm = GaussianMixture(n_components=N_CLUSTERS, covariance_type='full', random_state=42)
    cluster_labels = gmm.fit_predict(z_reduced)
    print(f"✓ GMM clustering complete: {N_CLUSTERS} clusters")
    metadata_results['clustering_implementation'] = 'sklearn.mixture'

elif CLUSTERING_METHOD == 'dbscan':
    # DBSCAN parameters (adjust these!)
    eps = 3.0  # Neighborhood radius
    min_samples = 5  # Minimum points to form cluster
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    cluster_labels = dbscan.fit_predict(z_reduced)
    
    n_clusters_found = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)
    
    print(f"✓ DBSCAN complete:")
    print(f"  Clusters found: {n_clusters_found}")
    print(f"  Noise points: {n_noise}")
    print(f"  (eps={eps}, min_samples={min_samples})")
    metadata_results['clustering_implementation'] = 'sklearn.cluster'

# Print cluster distribution
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Cluster {label}: {count} images")

In [ ]:
# ============================================================================
# STEP 5: Visualize with t-SNE
# ============================================================================

from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
z_2d = tsne.fit_transform(z_reduced)

plt.figure(figsize=(12, 12))
scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1], c=cluster_labels, 
                     cmap='tab10', s=100, alpha=0.7, edgecolors='black', linewidth=0.5)
plt.colorbar(scatter, label='Cluster')
plt.title(f'{CLUSTERING_METHOD.upper()} Clustering Results (t-SNE visualization)\n{len(set(cluster_labels))} clusters', 
          fontsize=16)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.show()

# ============================================================================
# STEP 6: Examine images in each cluster
# ============================================================================

print("\n" + "="*80)
print("CLUSTER VERIFICATION - Sample images from each cluster:")
print("="*80)

for cluster_id in sorted(set(cluster_labels)):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    print(f"\nCluster {cluster_id}: {len(cluster_indices)} images")
    
    # Print first 5 image paths
    for idx in cluster_indices[:5]:
        print(f"  {samples['image_paths'][idx]}")

In [ ]:
metadata_results['clustering_implementation']

### Store metadata:

In [ ]:
# Store clustering params:

metadata_results['clustering_name'] = CLUSTERING_METHOD
#metadata_results['clustering_implementation'] = 'sklearn.cluster'
clustering_params['n_clusters'] = N_CLUSTERS
metadata_results['clustering_params'] = clustering_params

### Link cluster data from latent space to the respective sample images with image_ids:

In [ ]:
image_indices = list(range(len(cluster_labels)))

cluster_data = pd.DataFrame({'image_index': image_indices,
                            'cluster_label': cluster_labels, 
                             'class_label': samples['labels'], 
                             'image_paths': samples['image_paths']})
cluster_data.head()

In [ ]:
import re
def extract_number(filepath):
    match = re.search(r'(\d+)\.jpg$', filepath)
    if not match:
        raise ValueError(f"No number found at end of filename: {filepath}")
    return int(match.group(1))



In [ ]:
cluster_data['image_id'] = cluster_data['image_paths'].apply(extract_number)
cluster_data.head()

In [ ]:
#cluster_data[cluster_data.image_id==2370502].image_paths[0]

### Add the features used in the clustering to the cluster_data:

In [ ]:
for i in range(0, z_reduced.shape[1]): 
    feature_name_i = 'feature_' + str(i)
    feature_i = z_reduced[:, i]
    cluster_data[feature_name_i] = feature_i

In [ ]:
cluster_data.head()

### Extract file source from file paths and add to the cluster data:

In [ ]:
file_sources

In [ ]:
cluster_data.shape

In [ ]:
file_sources_to_add = ['giub'] * cluster_data.shape[0]

In [ ]:
cluster_data['file_source'] = file_sources_to_add
cluster_data.head()

In [ ]:
print(cluster_data[cluster_data.file_source==file_sources[0]].shape)
#print(cluster_data[cluster_data.file_source==file_sources[1]].shape)
print(set(cluster_data[cluster_data.file_source==file_sources[0]].file_source))
#print(set(cluster_data[cluster_data.file_source==file_sources[1]].file_source))
print(cluster_data[cluster_data.file_source==file_sources[0]].image_paths.iloc[0])
#print(cluster_data[cluster_data.file_source==file_sources[1]].file_source.iloc[0])
print(cluster_data[cluster_data.file_source==file_sources[0]].image_paths.iloc[20])
#print(cluster_data[cluster_data.file_source==file_sources[1]].file_source.iloc[20])

### Store metadata: 

In [ ]:
def store_duration(time_analysis_dict, time_analysis_for_df_dict, analysis_name, duration,
                  timestamp_start_is_photo_analysis,
                  timestamp_end_is_photo_analysis):
    print(time_analysis_dict)
    print(time_analysis_for_df_dict)
    time_analysis_dict[analysis_name] = {}
    time_analysis_dict[analysis_name]['duration_str'] = f"Analysis took: {duration}"
    time_analysis_dict[analysis_name]['duration_seconds'] = total_seconds
    time_analysis_dict[analysis_name]['duration_seconds_str'] = f"Analysis took: {total_seconds:.2f} seconds"
    time_analysis_dict[analysis_name]['duration_minutes'] = total_seconds/60
    time_analysis_dict[analysis_name]['duration_minutes_str'] = f"Analysis took: {total_seconds/60:.2f} minutes"
    time_analysis_dict[analysis_name]['time_stamp_start'] = timestamp_start_is_photo_analysis
    time_analysis_dict[analysis_name]['time_stamp_end'] = timestamp_end_is_photo_analysis

    time_analysis_for_df_dict['analysis_name'].append(analysis_name)
    print(time_analysis_for_df_dict)
    time_analysis_for_df_dict['time_stamp_start'].append(timestamp_start_is_photo_analysis)
    time_analysis_for_df_dict['duration_str'].append(f"Analysis took: {duration}")
    time_analysis_for_df_dict['duration_seconds'].append(total_seconds)
    time_analysis_for_df_dict['duration_seconds_str'].append(f"Analysis took: {total_seconds:.2f} seconds")
    time_analysis_for_df_dict['duration_minutes'].append(total_seconds/60)
    time_analysis_for_df_dict['duration_minutes_str'].append(f"Analysis took: {total_seconds/60:.2f} minutes")
    print(time_analysis_for_df_dict)
    return time_analysis_dict, time_analysis_for_df_dict

In [ ]:
time_analyses = {}
time_analyses_clustering_pipeline = {}
time_analyses_clustering_pipeline['analysis_name'] = []
time_analyses_clustering_pipeline['time_stamp_start'] = []
time_analyses_clustering_pipeline['duration_str'] = []
time_analyses_clustering_pipeline['duration_seconds'] = []
time_analyses_clustering_pipeline['duration_seconds_str'] = []
time_analyses_clustering_pipeline['duration_minutes'] = []
time_analyses_clustering_pipeline['duration_minutes_str'] = []


In [ ]:
timestamp_end_clustering_pipeline = pd.Timestamp.now()


duration = timestamp_end_clustering_pipeline - timestamp_start_clustering_pipeline

total_seconds = duration.total_seconds()
print(total_seconds)


workflow_name = 'clustering_pipeline'

# Store information about duration: 
time_analyses, time_analyses_clustering_pipeline = store_duration(time_analyses, time_analyses_clustering_pipeline, workflow_name, duration, timestamp_start_clustering_pipeline, timestamp_end_clustering_pipeline)
#time_analyses, time_analyses_yolo                = store_duration(time_analyses, 
#                                                    time_analyses_yolo, 
#                                                    analysis_name, duration, 
#                                                    timestamp_start_pers_yolo, 
#                                                    timestamp_end_pers_yolo)
timestamp_id = timestamp_start_clustering_pipeline.strftime('%Y%m%d_%H%M%S') 


In [ ]:
timestamp_start_clustering_pipeline

In [ ]:
timestamp_start_clustering_pipeline

In [ ]:
timestamp_id

In [ ]:
metadata_results

In [ ]:
metadata_results['run_timestamp'] = timestamp_id
# Store results: 
metadata_results['cluster_data'] = cluster_data

### Save results and metadata:

In [ ]:
import pickle
filename = 'results_' + workflow_name + '_' + timestamp_id + '.pkl'
file_path = os.path.join(data_path, filename)
# Save
with open(file_path, 'wb') as f:
    pickle.dump(metadata_results, f)

In [ ]:
with open(file_path, 'rb') as f:
    metadata_results = pickle.load(f)
cluster_data = metadata_results['cluster_data']

### Save times data: 

In [ ]:

# Define file name: 

time_analyses_clustering_file_name = 'times_' + workflow_name + '_' + timestamp_id + '.pkl'

# Save dictionary:
time_analyses_clustering_output_path = os.path.join(data_path, time_analyses_clustering_file_name)
with open(time_analyses_clustering_output_path, 'wb') as f:
   pickle.dump(time_analyses_clustering_pipeline, f)

# Reload saved dictionary to check if saving worked:
with open(time_analyses_clustering_output_path, 'rb') as f:
   reloaded_time_analyses_clustering_pipeline = pickle.load(f)

# Check if original and reloaded dictionary are the same:
print(len(time_analyses_clustering_pipeline))
print(type(time_analyses_clustering_pipeline))
print(type(reloaded_time_analyses_clustering_pipeline))
print(len(reloaded_time_analyses_clustering_pipeline))

print(time_analyses_clustering_pipeline.keys() == reloaded_time_analyses_clustering_pipeline.keys())


In [ ]:
def calculate_label_proportion(cluster_selected, label_name):
    label_sum = sum(cluster_selected[label_name])
    label_proportion = label_sum/cluster_selected.shape[0]
    return label_proportion

In [ ]:
label_proportions_in_clusters = {}

for cluster_label in cluster_data.cluster_label:
    cluster_selected = cluster_data[cluster_data.cluster_label == cluster_label]
    label_proportion = calculate_label_proportion(cluster_selected, 'class_label')
    label_proportions_in_clusters[cluster_label] = label_proportion
    

In [ ]:
import matplotlib.pyplot as plt

def plot_bar_chart(data_dict, title="Bar Chart", xlabel="Keys", ylabel="Values", figsize=(12, 6)):
   """
   Create a bar plot from a dictionary with sorted keys.
   
   Args:
       data_dict: Dictionary with keys as x-axis labels and values as bar heights
       title: Plot title
       xlabel: X-axis label
       ylabel: Y-axis label
       figsize: Figure size tuple
   """
   # Sort the dictionary by keys
   sorted_items = sorted(data_dict.items())
   keys = [item[0] for item in sorted_items]
   values = [item[1] for item in sorted_items]
   
   # Create the bar plot
   plt.figure(figsize=figsize)
   plt.bar(keys, values)
   plt.title(title)
   plt.xlabel(xlabel)
   plt.ylabel(ylabel)
   plt.grid(axis='y', alpha=0.3)
   
   # Ensure all keys are shown as tick labels
   plt.xticks(keys)
   
   # Rotate x-axis labels if there are many keys
   if len(keys) > 10:
       plt.xticks(rotation=45)
   
   plt.tight_layout()
   plt.show()

## Usage with your data:
#data = {11: 0.5, 0: 0.2692307692307692, 9: 0.6153846153846154, 2: 0.6666666666666666, 
#       5: 0.43478260869565216, 14: 0.36666666666666664, 4: 0.6, 8: 0.7083333333333334, 
#       3: 0.46153846153846156, 15: 0.18181818181818182, 1: 0.5833333333333334, 
#       13: 0.6666666666666666, 16: 0.5, 7: 0.3333333333333333, 12: 0.8, 
#       10: 0.5833333333333334, 6: 0.3333333333333333}
#
#plot_bar_chart(data, title="Data Distribution", xlabel="Categories", ylabel="Values")

In [ ]:
import matplotlib.pyplot as plt

def plot_bar_chart(data_dict, title="Bar Chart", xlabel="Keys", ylabel="Values", figsize=(12, 6), 
                   title_fontsize=16, label_fontsize=14, tick_fontsize=12):
    """
    Create a bar plot from a dictionary with sorted keys.
    
    Args:
        data_dict: Dictionary with keys as x-axis labels and values as bar heights
        title: Plot title
        xlabel: X-axis label
        ylabel: Y-axis label
        figsize: Figure size tuple
        title_fontsize: Font size for the title
        label_fontsize: Font size for axis labels
        tick_fontsize: Font size for tick labels
    """
    # Sort the dictionary by keys
    sorted_items = sorted(data_dict.items())
    keys = [item[0] for item in sorted_items]
    values = [item[1] for item in sorted_items]
    
    # Create the bar plot
    plt.figure(figsize=figsize)
    plt.bar(range(len(keys)), values)  # Use range for x-positions
    plt.title(title, fontsize=title_fontsize)
    plt.xlabel(xlabel, fontsize=label_fontsize)
    plt.ylabel(ylabel, fontsize=label_fontsize)
    plt.grid(axis='y', alpha=0.3)
    
    # Set x-axis ticks and labels properly for categorical data
    plt.xticks(range(len(keys)), keys, fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    
    # Rotate x-axis labels if there are many keys
    if len(keys) > 10:
        plt.xticks(range(len(keys)), keys, rotation=45, fontsize=tick_fontsize)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_bar_chart(label_proportions_in_clusters, xlabel='clusters', ylabel='proportion of class label', label_fontsize=18, tick_fontsize=18, figsize=(17, 10))


In [ ]:
def plot_images_per_cluster(cluster_data, image_dir, n_images, label_columns):
    """
    Plot images per cluster with their labels and cluster id.
    
    Args:
        cluster_data: DataFrame with cluster_label, image_paths, and label columns
        image_dir: directory containing the images (not used directly, paths are in cluster_data)
        n_images: maximum number of images to plot per cluster
        label_columns: list of label column names to display
    """
    unique_clusters = sorted(set(cluster_data.cluster_label))
    
    for cluster_id in unique_clusters:
        cluster_subset = cluster_data[cluster_data.cluster_label == cluster_id].reset_index(drop=True)
        
        # Plot as many images as available up to n_images
        n_to_plot = min(n_images, len(cluster_subset))
        
        print(f"\nCluster {cluster_id} — showing {n_to_plot} of {len(cluster_subset)} images")
        
        # Plot in rows of 4
        n_cols = 4
        n_rows = int(np.ceil(n_to_plot / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
        axes = np.array(axes).reshape(-1)  # Flatten to 1D for easy indexing
        
        for i in range(n_to_plot):
            image_path = cluster_subset.image_paths[i]
            image = Image.open(image_path).convert('L')
            image = image.resize(target_size, Image.Resampling.LANCZOS)
            
            # Build title from label values
            label_str = '\n'.join([f"{col}: {cluster_subset[col][i]}" for col in label_columns])
            title = f"Cluster {cluster_id}\n{os.path.basename(image_path)}\n{label_str}"
            
            axes[i].imshow(image, cmap='gray')
            axes[i].set_title(title, fontsize=7)
            axes[i].axis('off')
        
        # Hide unused axes
        for i in range(n_to_plot, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()


# User defined: number of images to plot per cluster
N_IMAGES_TO_PLOT = 90

plot_images_per_cluster(cluster_data, image_dir, N_IMAGES_TO_PLOT, ['class_label'])

In [ ]:
# Visualize the clusters
plt.figure(figsize=(8, 6))
scatter = plt.scatter(cluster_data.feature_0, cluster_data.feature_1, c=cluster_data.cluster_label, cmap='viridis', s=100, alpha=0.7)

# Add cluster labels at cluster centers
unique_clusters = np.unique(cluster_data.cluster_label)
for cluster_id in unique_clusters:
    # Find points belonging to this cluster
    cluster_points = cluster_data[cluster_data.cluster_label == cluster_id]
    
    # Calculate centroid of this cluster
    center_x = cluster_points.feature_0.mean()
    center_y = cluster_points.feature_1.mean()
    
    # Add text label at the center
    plt.text(center_x, center_y, str(int(cluster_id)), 
             fontsize=12, fontweight='bold', ha='center', va='center',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.title('K-means Clustering Results')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

# Add colorbar to show cluster mapping
plt.colorbar(scatter, label='Cluster ID')

plt.grid(True, alpha=0.3)
plt.show()



    

In [ ]:
samples_indexed = {}
for idx in cluster_data.image_index:
    image = samples['images'][idx][0]
    #print(image.shape)
    samples_indexed[idx] = image
    

In [ ]:
sample_history[59]['y'][1][0].shape

In [ ]:
reconstructions_indexed = {}
for idx in cluster_data.image_index:
    reconstruction = sample_history[59]['y'][idx][0]
    reconstructions_indexed[idx] = reconstruction


In [ ]:
cluster_data.shape

In [ ]:
def plot_images_from_cluster(cluster_data, cluster_label, image_data_indexed):
    cluster_selected = cluster_data[cluster_data.cluster_label == cluster_label]
    print(cluster_selected.head())
    
    # Group images into rows of 3
    for i in range(0, len(cluster_selected.image_index), 3):
       # Get up to 3 images for this row
       row_indices = cluster_selected.image_index[i:i+3]
       
       # Create subplot with 1 row and number of images in this row
       fig, axes = plt.subplots(1, len(row_indices), figsize=(15, 5))
       
       # Handle case where there's only one image (axes won't be a list)
       if len(row_indices) == 1:
           axes = [axes]
       
       for j, image_index in enumerate(row_indices):
           #image = samples_indexed[image_index]
           image = image_data_indexed[image_index]
           print(f"Row {i//3 + 1}, Image {j+1}: {image.shape}")
           
           axes[j].imshow(image, cmap='gray')
           axes[j].axis('off')  # Remove axis labels
           axes[j].set_title(f'Index: {image_index}')
       
       plt.tight_layout()
       plt.show()

In [ ]:
plot_images_from_cluster(cluster_data, 0, reconstructions_indexed)

In [ ]:
plot_images_from_cluster(cluster_data, 1, reconstructions_indexed)

In [ ]:
plot_images_from_cluster(cluster_data, 1, reconstructions_indexed)

In [ ]:
plot_images_from_cluster(cluster_data, 2, reconstructions_indexed)

In [ ]:
latent_space = sample_history[59]['z'][0]

In [ ]:
type(latent_space)
latent_space.shape

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(perplexity= 1, init="pca", learning_rate="auto", random_state=42)
X_valid_2D = tsne.fit_transform(latent_space)


In [ ]:
# Record paramters:
metadata_results['dim_reduction_name'] = 'TSNE',
metadata_results['dim_reduction_implementation'] = 'sklearn.manifold.TSNE'
metadata_results['dim_reduction_params'] ={ 'perplexity': 1, 'init': 'pca'}

In [ ]:
plt.scatter(X_valid_2D[:, 0], X_valid_2D[:, 1], s=10, cmap="tab10")
plt.show()

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Example array with shape (10, 2)
# Replace this with your actual data
data = X_valid_2D.copy()


inertia = []
k_range = range(1, 20)  # Test from 1 to 9 clusters

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(data)
    inertia.append(kmeans.inertia_)

# Plot the elbow curve
plt.figure(figsize=(8, 6))
plt.plot(k_range, inertia, 'o-', color='blue')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
data.shape

In [ ]:
z_reduced.shape

In [ ]:
type(data)

In [ ]:
# Specify the number of clusters (k)
# You can adjust this based on your needs
k = 12

# Initialize and fit the k-means model
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
kmeans.fit(data)

# Get cluster assignments for each data point
labels = kmeans.labels_

# Get the coordinates of the cluster centers
centers = kmeans.cluster_centers_

# Print results
print("Cluster assignments:", labels)
print("Cluster centers:", centers)